In [9]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import pandas as pd; print(pd.__version__)"
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

Return code: 0
Output: 3.1.1

Error: 


In [10]:
import sys

print("Pandas loaded:", "pandas" in sys.modules)
print("Pandas-related modules:")

for name in sys.modules:
    if name == "pandas" or name.startswith("pandas."):
        print(name)

Pandas loaded: False
Pandas-related modules:
pandas.compat._constants
pandas.util
pandas.util.version
pandas.compat.numpy
pandas.compat.pyarrow
pandas.compat
pandas.util._exceptions
pandas._config.config
pandas._config.dates
pandas._config.display
pandas._config
pandas.core
pandas._libs.pandas_parser
pandas._libs.pandas_datetime
pandas._libs._cyutility
pandas._libs.tslibs.ccalendar
pandas._libs.tslibs.np_datetime
pandas._libs.tslibs.dtypes
pandas._libs.tslibs.base
pandas._libs.tslibs.nattype
pandas.compat._optional
pandas._libs.tslibs.timezones
pandas._libs.properties
pandas.util._decorators
pandas._config.localization
pandas._libs.tslibs.fields
pandas._libs.tslibs.timedeltas
pandas._libs.tslibs.tzconversion
pandas._libs.tslibs.timestamps
pandas._libs.tslibs.offsets


In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)
print("Pandas imported successfully!")

Pandas version: 3.1.1
Pandas imported successfully!


In [12]:
import sys

for name in [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "pyarrow",
    "bottleneck",
    "numexpr"
]:
    print(name, "->", name in sys.modules)

numpy -> True
pandas -> False
matplotlib -> False
seaborn -> False
pyarrow -> False
bottleneck -> False
numexpr -> False


In [13]:
import numpy as np

print("NumPy version:", np.__version__)
print("NumPy location:", np.__file__)
print("NumPy C API:", hasattr(np, "_core"))
print("Datetime C API:", hasattr(np, "datetime_data"))

NumPy version: 2.5.4
NumPy location: d:\RR-dhanush_sj\.venv\Lib\site-packages\numpy\__init__.py
NumPy C API: True
Datetime C API: True


In [15]:
import subprocess
import sys

test_code = """
import numpy as np
import pandas as pd
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Both imports successful")
"""

result = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

Return code: 0
Output: NumPy: 2.5.4
Pandas: 3.1.1
Both imports successful

Error: 


In [4]:
import pandas as pd
import numpy as np
import os
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Imports successful!")

Pandas: 3.1.1
NumPy: 2.5.4
Imports successful!


In [5]:
DATA_PATH = "../outputs/cleaned_data/"
PRESSURE_PATH = "../outputs/pressure_analysis/"
OUTPUT_PATH = "../outputs/batter_analysis/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

ball_by_ball = pd.read_csv(
    DATA_PATH + "ball_by_ball_cleaned.csv"
)

pressure = pd.read_csv(
    PRESSURE_PATH + "ipl_chase_pressure_deliveries.csv"
)

print("Ball-by-ball shape:", ball_by_ball.shape)
print("Pressure dataset shape:", pressure.shape)

Ball-by-ball shape: (200999, 33)
Pressure dataset shape: (41700, 51)


In [6]:
print("Ball-by-ball columns:")
print(ball_by_ball.columns.tolist())

print("\nPressure dataset columns:")
print(pressure.columns.tolist())

Ball-by-ball columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

Pressure dataset columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed',

In [7]:
delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num"
]

# Check duplicate delivery keys
print("Ball-by-ball duplicate keys:",
      ball_by_ball.duplicated(delivery_key).sum())

print("Pressure duplicate keys:",
      pressure.duplicated(delivery_key).sum())

# Check whether every pressure delivery exists in ball-by-ball data
matching_check = pressure.merge(
    ball_by_ball[delivery_key],
    on=delivery_key,
    how="left",
    indicator=True,
    validate="one_to_one"
)

print("\nPressure delivery matching:")
print(matching_check["_merge"].value_counts())

Ball-by-ball duplicate keys: 0
Pressure duplicate keys: 0

Pressure delivery matching:
_merge
both          41700
left_only         0
right_only        0
Name: count, dtype: int64


In [8]:
batter_pressure = ball_by_ball.merge(
    pressure[
        delivery_key + [
            "combined_pressure_score",
            "pressure_category",
            "rrr_pressure_score",
            "wicket_pressure_score",
            "time_pressure_score"
        ]
    ],
    on=delivery_key,
    how="inner",
    validate="one_to_one"
)

print("Merged dataset shape:", batter_pressure.shape)

print("\nPressure category distribution:")
print(batter_pressure["pressure_category"].value_counts())

print("\nUnique batters:", batter_pressure["striker_id"].nunique())
print("Unique matches:", batter_pressure["match_id"].nunique())

Merged dataset shape: (41700, 38)

Pressure category distribution:
pressure_category
Moderate    18130
Low         14364
High         6362
Extreme      2844
Name: count, dtype: int64

Unique batters: 311
Unique matches: 363


In [9]:
batter_basic_stats = (
    batter_pressure
    .groupby(["striker_id", "striker_name"])
    .agg(
        balls_faced=("striker_id", "count"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("dismissed_player_id", "count"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

batter_basic_stats["strike_rate"] = (
    batter_basic_stats["batter_runs"]
    / batter_basic_stats["balls_faced"]
    * 100
)

print("Number of batters:", len(batter_basic_stats))

print("\nTop 10 batters by runs in pressure observations:")
print(
    batter_basic_stats
    .sort_values("batter_runs", ascending=False)
    .head(10)
    .to_string(index=False)
)

Number of batters: 311

Top 10 batters by runs in pressure observations:
 striker_id     striker_name  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate
       6288     Shubman Gill         1130         1577    158     56          38              42   139.557522
         47      Virat Kohli          989         1451    154     48          26              34   146.713852
       6819 Yashasvi Jaiswal          846         1277    149     53          32              36   150.945626
       1699 Suryakumar Yadav          759         1250    133     57          30              34   164.690382
      23450    Sai Sudharsan          860         1214    130     37          28              31   141.162791
        813      Jos Buttler          765         1147    121     49          30              35   149.934641
        126     Rohit Sharma          827         1139    102     68          37              40   137.726723
        953     Lokesh Rahul          838      

In [10]:
batter_pressure["is_ball_faced"] = (
    ~batter_pressure["extras_type"].eq("wide")
)

batter_pressure["is_striker_dismissed"] = (
    batter_pressure["is_wicket"].fillna(False)
    & batter_pressure["dismissed_player_id"].eq(
        batter_pressure["striker_id"]
    )
)

batter_basic_stats = (
    batter_pressure
    .groupby(["striker_id", "striker_name"])
    .agg(
        delivery_records=("striker_id", "count"),
        balls_faced=("is_ball_faced", "sum"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_striker_dismissed", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

batter_basic_stats["strike_rate"] = (
    batter_basic_stats["batter_runs"]
    / batter_basic_stats["balls_faced"]
    * 100
)

print("Number of batters:", len(batter_basic_stats))

print("\nTop 10 batters by runs:")
print(
    batter_basic_stats
    .sort_values("batter_runs", ascending=False)
    .head(10)
    .to_string(index=False)
)

Number of batters: 311

Top 10 batters by runs:
 striker_id     striker_name  delivery_records  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate
       6288     Shubman Gill              1130         1098         1577    158     56          38              42   143.624772
         47      Virat Kohli               989          956         1451    154     48          24              34   151.778243
       6819 Yashasvi Jaiswal               846          817         1277    149     53          32              36   156.303550
       1699 Suryakumar Yadav               759          740         1250    133     57          29              34   168.918919
      23450    Sai Sudharsan               860          833         1214    130     37          28              31   145.738295
        813      Jos Buttler               765          743         1147    121     49          28              35   154.374159
        126     Rohit Sharma               827          

In [11]:
batter_pressure_stats = (
    batter_pressure
    .groupby(
        ["striker_id", "striker_name", "pressure_category"],
        observed=True
    )
    .agg(
        balls_faced=("is_ball_faced", "sum"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_striker_dismissed", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

batter_pressure_stats["strike_rate"] = (
    batter_pressure_stats["batter_runs"]
    / batter_pressure_stats["balls_faced"].replace(0, np.nan)
    * 100
)

batter_pressure_stats["dismissal_rate"] = (
    batter_pressure_stats["dismissals"]
    / batter_pressure_stats["balls_faced"].replace(0, np.nan)
    * 100
)

print("Batter pressure statistics shape:",
      batter_pressure_stats.shape)

print("\nSample results:")
print(batter_pressure_stats.head(15).to_string(index=False))

Batter pressure statistics shape: (749, 11)

Sample results:
 striker_id         striker_name pressure_category  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate  dismissal_rate
          8 Mahendra Singh Dhoni           Extreme          101          207     14     18           7              13   204.950495        6.930693
          8 Mahendra Singh Dhoni              High           95          110      7      4           1              14   115.789474        1.052632
          8 Mahendra Singh Dhoni          Moderate           18            9      0      0           0               5    50.000000        0.000000
         23        Piyush Chawla           Extreme           11           13      1      1           4               5   118.181818       36.363636
         23        Piyush Chawla              High           10           16      1      1           0               3   160.000000        0.000000
         28        Ishant Sharma           Extreme 

In [12]:
sample_size_summary = (
    batter_pressure_stats
    .groupby("pressure_category", observed=True)
    ["balls_faced"]
    .describe()
    .round(2)
)

print("Balls faced distribution by pressure category:")
print(sample_size_summary)

print("\nNumber of batters by minimum balls faced:")

for minimum in [10, 20, 30, 50, 100]:
    eligible = (
        batter_pressure_stats[
            batter_pressure_stats["balls_faced"] >= minimum
        ]
        .groupby("pressure_category", observed=True)
        ["striker_id"]
        .nunique()
    )

    print(f"\nMinimum {minimum} balls:")
    print(eligible)

Balls faced distribution by pressure category:
                   count   mean     std  min    25%   50%     75%    max
pressure_category                                                       
Extreme            199.0  13.34   15.19  1.0   3.00   9.0   18.00  101.0
High               214.0  28.20   36.41  1.0   5.00  12.0   35.50  166.0
Low                144.0  96.28  138.12  0.0  10.75  36.0  122.00  718.0
Moderate           192.0  91.03  110.88  1.0  10.75  40.5  141.75  590.0

Number of batters by minimum balls faced:

Minimum 10 balls:
pressure_category
Extreme      95
High        124
Low         111
Moderate    149
Name: striker_id, dtype: int64

Minimum 20 balls:
pressure_category
Extreme      42
High         85
Low          94
Moderate    120
Name: striker_id, dtype: int64

Minimum 30 balls:
pressure_category
Extreme      14
High         65
Low          84
Moderate    109
Name: striker_id, dtype: int64

Minimum 50 balls:
pressure_category
Extreme      8
High        35
Low      

In [13]:
valid_batter_samples = batter_pressure_stats[
    batter_pressure_stats["balls_faced"] > 0
].copy()

thresholds = [10, 20, 30, 50, 100]

eligibility_results = []

for category in ["Low", "Moderate", "High", "Extreme"]:
    category_data = valid_batter_samples[
        valid_batter_samples["pressure_category"] == category
    ]

    row = {"pressure_category": category}

    for minimum in thresholds:
        row[f"{minimum}_balls"] = (
            category_data[
                category_data["balls_faced"] >= minimum
            ]["striker_id"].nunique()
        )

    eligibility_results.append(row)

eligibility_table = pd.DataFrame(eligibility_results)

print(eligibility_table.to_string(index=False))

pressure_category  10_balls  20_balls  30_balls  50_balls  100_balls
              Low       111        94        84        63         40
         Moderate       149       120       109        92         62
             High       124        85        65        35         19
          Extreme        95        42        14         8          1


In [14]:
MIN_BALLS = 20

eligible_batter_stats = valid_batter_samples[
    valid_batter_samples["balls_faced"] >= MIN_BALLS
].copy()

print("Minimum balls required:", MIN_BALLS)

print("\nEligible batters by pressure category:")

print(
    eligible_batter_stats
    .groupby("pressure_category", observed=True)["striker_id"]
    .nunique()
)

print("\nTotal eligible batter-category records:",
      len(eligible_batter_stats))

Minimum balls required: 20

Eligible batters by pressure category:
pressure_category
Extreme      42
High         85
Low          94
Moderate    120
Name: striker_id, dtype: int64

Total eligible batter-category records: 341


In [15]:
eligible_batter_stats["boundary_runs"] = (
    eligible_batter_stats["fours"] * 4
    + eligible_batter_stats["sixes"] * 6
)

eligible_batter_stats["boundary_percentage"] = (
    eligible_batter_stats["boundary_runs"]
    / eligible_batter_stats["batter_runs"].replace(0, np.nan)
    * 100
)

eligible_batter_stats["runs_per_match"] = (
    eligible_batter_stats["batter_runs"]
    / eligible_batter_stats["matches_played"]
)

print("Batter reliability indicators:")
print(
    eligible_batter_stats[
        [
            "striker_name",
            "pressure_category",
            "balls_faced",
            "batter_runs",
            "strike_rate",
            "dismissal_rate",
            "boundary_percentage",
            "runs_per_match"
        ]
    ].head(15).round(2).to_string(index=False)
)

Batter reliability indicators:
        striker_name pressure_category  balls_faced  batter_runs  strike_rate  dismissal_rate  boundary_percentage  runs_per_match
Mahendra Singh Dhoni           Extreme          101          207       204.95            6.93                79.23           15.92
Mahendra Singh Dhoni              High           95          110       115.79            1.05                47.27            7.86
         Virat Kohli               Low          546          849       155.49            1.83                67.84           25.73
         Virat Kohli          Moderate          403          590       146.40            3.23                54.24           25.65
      Faf du Plessis               Low          340          528       155.29            4.12                70.83           22.00
      Faf du Plessis          Moderate          128          202       157.81            6.25                67.33           20.20
     Wriddhiman Saha               Low          301 

In [16]:
print("Boundary runs greater than batter runs:")
print(
    (
        eligible_batter_stats["boundary_runs"]
        > eligible_batter_stats["batter_runs"]
    ).sum()
)

print("\nNegative batter runs:")
print((eligible_batter_stats["batter_runs"] < 0).sum())

print("\nNegative dismissal counts:")
print((eligible_batter_stats["dismissals"] < 0).sum())

print("\nStrike rate missing values:")
print(eligible_batter_stats["strike_rate"].isna().sum())

print("\nDismissal rate missing values:")
print(eligible_batter_stats["dismissal_rate"].isna().sum())

Boundary runs greater than batter runs:
0

Negative batter runs:
0

Negative dismissal counts:
0

Strike rate missing values:
0

Dismissal rate missing values:
0


In [17]:
batter_basic_stats.to_csv(
    OUTPUT_PATH + "batter_basic_stats.csv",
    index=False
)

batter_pressure_stats.to_csv(
    OUTPUT_PATH + "batter_pressure_category_stats.csv",
    index=False
)

eligible_batter_stats.to_csv(
    OUTPUT_PATH + "eligible_batter_reliability_stats.csv",
    index=False
)

print("Batter analysis files saved successfully.")

Batter analysis files saved successfully.


In [18]:
batter_pressure["pressure_group"] = batter_pressure[
    "pressure_category"
].map({
    "Low": "Normal",
    "Moderate": "Normal",
    "High": "High",
    "Extreme": "High"
})

pressure_group_stats = (
    batter_pressure.groupby(
        ["striker_id", "striker_name", "pressure_group"],
        observed=True
    )
    .agg(
        balls_faced=("is_ball_faced", "sum"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_striker_dismissed", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

pressure_group_stats["strike_rate"] = (
    pressure_group_stats["batter_runs"]
    / pressure_group_stats["balls_faced"].replace(0, np.nan)
    * 100
)

pressure_group_stats["dismissal_rate"] = (
    pressure_group_stats["dismissals"]
    / pressure_group_stats["balls_faced"].replace(0, np.nan)
    * 100
)

print("Pressure group counts:")
print(pressure_group_stats["pressure_group"].value_counts())

print("\nSample results:")
print(pressure_group_stats.head(10).to_string(index=False))

Pressure group counts:
pressure_group
High      259
Normal    203
Name: count, dtype: int64

Sample results:
 striker_id         striker_name pressure_group  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate  dismissal_rate
          8 Mahendra Singh Dhoni           High          196          317     21     22           8              17   161.734694        4.081633
          8 Mahendra Singh Dhoni         Normal           18            9      0      0           0               5    50.000000        0.000000
         23        Piyush Chawla           High           21           29      2      2           4               6   138.095238       19.047619
         28        Ishant Sharma           High            7            1      0      0           0               2    14.285714        0.000000
         47          Virat Kohli           High            7           12      2      0           1               1   171.428571       14.285714
         47          

In [19]:
pressure_comparison = pressure_group_stats.pivot(
    index=["striker_id", "striker_name"],
    columns="pressure_group",
    values=[
        "balls_faced",
        "batter_runs",
        "dismissals",
        "matches_played",
        "strike_rate",
        "dismissal_rate"
    ]
)

pressure_comparison.columns = [
    f"{metric}_{group.lower()}"
    for metric, group in pressure_comparison.columns
]

pressure_comparison = pressure_comparison.reset_index()

eligible_pressure_comparison = pressure_comparison[
    (pressure_comparison["balls_faced_normal"] >= 20)
    & (pressure_comparison["balls_faced_high"] >= 20)
].copy()

print("Eligible batters in both groups:",
      len(eligible_pressure_comparison))

print("\nSample comparison:")
print(
    eligible_pressure_comparison[
        [
            "striker_name",
            "balls_faced_normal",
            "balls_faced_high",
            "strike_rate_normal",
            "strike_rate_high",
            "dismissal_rate_normal",
            "dismissal_rate_high"
        ]
    ].head(15).to_string(index=False)
)

Eligible batters in both groups: 69

Sample comparison:
       striker_name  balls_faced_normal  balls_faced_high  strike_rate_normal  strike_rate_high  dismissal_rate_normal  dismissal_rate_high
     Dinesh Karthik                29.0             158.0          134.482759        184.177215               3.448276             5.063291
    Ravindra Jadeja               162.0             188.0          126.543210        151.595745               3.086420             3.723404
      Manish Pandey               150.0              33.0          108.000000        166.666667               6.000000             6.060606
       David Warner               438.0              54.0          144.063927        144.444444               3.881279             5.555556
Ravichandran Ashwin                62.0              33.0           95.161290        181.818182               8.064516            15.151515
     Kieron Pollard                34.0              63.0           91.176471        111.111111         

In [20]:
eligible_pressure_comparison["strike_rate_change_pct"] = (
    (
        eligible_pressure_comparison["strike_rate_high"]
        - eligible_pressure_comparison["strike_rate_normal"]
    )
    / eligible_pressure_comparison["strike_rate_normal"].replace(0, np.nan)
) * 100

eligible_pressure_comparison["dismissal_rate_change"] = (
    eligible_pressure_comparison["dismissal_rate_high"]
    - eligible_pressure_comparison["dismissal_rate_normal"]
)

eligible_pressure_comparison["high_pressure_runs_per_match"] = (
    eligible_pressure_comparison["batter_runs_high"]
    / eligible_pressure_comparison["matches_played_high"].replace(0, np.nan)
)

print(
    eligible_pressure_comparison[
        [
            "striker_name",
            "strike_rate_normal",
            "strike_rate_high",
            "strike_rate_change_pct",
            "dismissal_rate_normal",
            "dismissal_rate_high",
            "dismissal_rate_change",
            "batter_runs_high",
            "matches_played_high",
            "high_pressure_runs_per_match"
        ]
    ].head(15).to_string(index=False)
)

       striker_name  strike_rate_normal  strike_rate_high  strike_rate_change_pct  dismissal_rate_normal  dismissal_rate_high  dismissal_rate_change  batter_runs_high  matches_played_high  high_pressure_runs_per_match
     Dinesh Karthik          134.482759        184.177215               36.952288               3.448276             5.063291               1.615015             291.0                 11.0                     26.454545
    Ravindra Jadeja          126.543210        151.595745               19.797613               3.086420             3.723404               0.636985             285.0                 16.0                     17.812500
      Manish Pandey          108.000000        166.666667               54.320988               6.000000             6.060606               0.060606              55.0                  2.0                     27.500000
       David Warner          144.063927        144.444444                0.264131               3.881279             5.555556   

In [21]:
eligible_pressure_comparison["strike_rate_change_pct"] = (
    (
        eligible_pressure_comparison["strike_rate_high"]
        - eligible_pressure_comparison["strike_rate_normal"]
    )
    / eligible_pressure_comparison["strike_rate_normal"].replace(0, np.nan)
) * 100

eligible_pressure_comparison["dismissal_rate_change"] = (
    eligible_pressure_comparison["dismissal_rate_high"]
    - eligible_pressure_comparison["dismissal_rate_normal"]
)

eligible_pressure_comparison["high_pressure_runs_per_match"] = (
    eligible_pressure_comparison["batter_runs_high"]
    / eligible_pressure_comparison["matches_played_high"].replace(0, np.nan)
)

print(
    eligible_pressure_comparison[
        [
            "striker_name",
            "strike_rate_normal",
            "strike_rate_high",
            "strike_rate_change_pct",
            "dismissal_rate_normal",
            "dismissal_rate_high",
            "dismissal_rate_change",
            "batter_runs_high",
            "matches_played_high",
            "high_pressure_runs_per_match"
        ]
    ].head(15).to_string(index=False)
)

       striker_name  strike_rate_normal  strike_rate_high  strike_rate_change_pct  dismissal_rate_normal  dismissal_rate_high  dismissal_rate_change  batter_runs_high  matches_played_high  high_pressure_runs_per_match
     Dinesh Karthik          134.482759        184.177215               36.952288               3.448276             5.063291               1.615015             291.0                 11.0                     26.454545
    Ravindra Jadeja          126.543210        151.595745               19.797613               3.086420             3.723404               0.636985             285.0                 16.0                     17.812500
      Manish Pandey          108.000000        166.666667               54.320988               6.000000             6.060606               0.060606              55.0                  2.0                     27.500000
       David Warner          144.063927        144.444444                0.264131               3.881279             5.555556   

In [22]:
print("High-pressure matches distribution:")
print(
    eligible_pressure_comparison["matches_played_high"]
    .describe()
)

print("\nBatters by minimum high-pressure matches:")

for minimum in [3, 5, 10]:
    count = (
        eligible_pressure_comparison["matches_played_high"] >= minimum
    ).sum()

    print(f"At least {minimum} matches: {count} batters")

print("\nBatters with fewer than 5 high-pressure matches:")
print(
    eligible_pressure_comparison.loc[
        eligible_pressure_comparison["matches_played_high"] < 5,
        ["striker_name", "balls_faced_high", "matches_played_high"]
    ].to_string(index=False)
)

High-pressure matches distribution:
count    69.000000
mean      9.463768
std       5.771583
min       1.000000
25%       5.000000
50%       8.000000
75%      14.000000
max      28.000000
Name: matches_played_high, dtype: float64

Batters by minimum high-pressure matches:
At least 3 matches: 65 batters
At least 5 matches: 53 batters
At least 10 matches: 30 batters

Batters with fewer than 5 high-pressure matches:
        striker_name  balls_faced_high  matches_played_high
       Manish Pandey              33.0                  2.0
        David Warner              54.0                  3.0
       Ambati Rayudu              29.0                  4.0
         Jos Buttler              52.0                  4.0
           Jason Roy              21.0                  1.0
        David Willey              30.0                  2.0
         Nitish Rana              24.0                  4.0
      Mahipal Lomror              33.0                  4.0
       Rajat Patidar              37.0     

In [23]:
final_batter_pool = eligible_pressure_comparison[
    eligible_pressure_comparison["matches_played_high"] >= 5
].copy()

print("Final eligible batters:", len(final_batter_pool))

print("\nHigh-pressure matches:")
print(final_batter_pool["matches_played_high"].describe())

print("\nTop 10 by high-pressure runs:")
print(
    final_batter_pool[
        [
            "striker_name",
            "batter_runs_high",
            "balls_faced_high",
            "matches_played_high",
            "strike_rate_high",
            "dismissal_rate_high"
        ]
    ]
    .sort_values("batter_runs_high", ascending=False)
    .head(10)
    .to_string(index=False)
)

Final eligible batters: 53

High-pressure matches:
count    53.000000
mean     11.358491
std       5.244390
min       5.000000
25%       7.000000
50%      10.000000
75%      15.000000
max      28.000000
Name: matches_played_high, dtype: float64

Top 10 by high-pressure runs:
   striker_name  batter_runs_high  balls_faced_high  matches_played_high  strike_rate_high  dismissal_rate_high
  Rahul Tewatia             354.0             236.0                 28.0        150.000000             6.355932
    Rinku Singh             344.0             178.0                 15.0        193.258427             3.932584
      Tim David             322.0             178.0                 20.0        180.898876             6.179775
 Dinesh Karthik             291.0             158.0                 11.0        184.177215             5.063291
Ravindra Jadeja             285.0             188.0                 16.0        151.595745             3.723404
   David Miller             283.0             151.0 

In [24]:
reliability_columns = [
    "strike_rate_high",
    "dismissal_rate_high",
    "strike_rate_change_pct",
    "dismissal_rate_change",
    "matches_played_high"
]

print(
    final_batter_pool[reliability_columns]
    .describe()
    .round(2)
    .to_string()
)

print("\nMissing values:")
print(final_batter_pool[reliability_columns].isna().sum())

print("\nZero normal-pressure strike rates:")
print(
    (final_batter_pool["strike_rate_normal"] == 0).sum()
)

       strike_rate_high  dismissal_rate_high  strike_rate_change_pct  dismissal_rate_change  matches_played_high
count             53.00                53.00                   53.00                  53.00                53.00
mean             158.91                10.10                   17.00                   6.48                11.36
std               25.99                 4.12                   25.00                   4.37                 5.24
min              111.11                 3.72                  -30.61                  -4.53                 5.00
25%              145.20                 7.19                    0.42                   3.41                 7.00
50%              153.26                 9.26                   17.49                   6.03                10.00
75%              181.82                12.90                   28.74                   8.27                15.00
max              223.33                23.81                   91.06                  20.33     

In [25]:
correlation_columns = [
    "strike_rate_high",
    "dismissal_rate_high",
    "strike_rate_change_pct",
    "dismissal_rate_change",
    "matches_played_high"
]

correlation_matrix = final_batter_pool[
    correlation_columns
].corr().round(2)

print("Reliability indicator correlation matrix:")
print(correlation_matrix.to_string())

Reliability indicator correlation matrix:
                        strike_rate_high  dismissal_rate_high  strike_rate_change_pct  dismissal_rate_change  matches_played_high
strike_rate_high                    1.00                -0.25                    0.53                  -0.29                 0.04
dismissal_rate_high                -0.25                 1.00                   -0.15                   0.88                -0.50
strike_rate_change_pct              0.53                -0.15                    1.00                  -0.24                -0.15
dismissal_rate_change              -0.29                 0.88                   -0.24                   1.00                -0.31
matches_played_high                 0.04                -0.50                   -0.15                  -0.31                 1.00


In [26]:
score_data = final_batter_pool.copy()

# Higher strike rate is better
score_data["scoring_score"] = (
    score_data["strike_rate_high"].rank(pct=True) * 100
)

# Higher strike-rate improvement is better
score_data["retention_score"] = (
    score_data["strike_rate_change_pct"].rank(pct=True) * 100
)

# Lower dismissal rate is better
score_data["survival_score"] = (
    score_data["dismissal_rate_high"].rank(
        ascending=False,
        pct=True
    ) * 100
)

# More high-pressure matches provide stronger evidence
score_data["experience_score"] = (
    score_data["matches_played_high"].rank(pct=True) * 100
)

print(
    score_data[
        [
            "striker_name",
            "scoring_score",
            "retention_score",
            "survival_score",
            "experience_score"
        ]
    ].head(15).round(2).to_string(index=False)
)

       striker_name  scoring_score  retention_score  survival_score  experience_score
     Dinesh Karthik          79.25            81.13           94.34             53.77
    Ravindra Jadeja          47.17            56.60          100.00             84.91
Ravichandran Ashwin          75.47           100.00           11.32             36.79
     Kieron Pollard           1.89            64.15           81.13              8.49
       Sunil Narine         100.00            77.36            8.49             24.53
       Lokesh Rahul          11.32            11.32           16.98             24.53
       David Miller          84.91            83.02           73.58             92.45
      Andre Russell          22.64             3.77           33.96             61.32
          Moeen Ali          33.96            88.68           13.21              8.49
   Suryakumar Yadav          56.60            16.98           20.75             61.32
       Sanju Samson          92.45            71.70   

In [27]:
score_data["reliability_score"] = (
    0.25 * score_data["scoring_score"]
    + 0.25 * score_data["retention_score"]
    + 0.25 * score_data["survival_score"]
    + 0.25 * score_data["experience_score"]
)

ranked_batters = score_data.sort_values(
    "reliability_score",
    ascending=False
).reset_index(drop=True)

ranked_batters["rank"] = (
    ranked_batters.index + 1
)

print("Top 15 Most Reliable Batters Under Pressure:")

print(
    ranked_batters[
        [
            "rank",
            "striker_name",
            "reliability_score",
            "strike_rate_high",
            "dismissal_rate_high",
            "strike_rate_change_pct",
            "matches_played_high"
        ]
    ].head(15).round(2).to_string(index=False)
)

Top 15 Most Reliable Batters Under Pressure:
 rank         striker_name  reliability_score  strike_rate_high  dismissal_rate_high  strike_rate_change_pct  matches_played_high
    1          Rinku Singh              91.27            193.26                 3.93                   63.87                 15.0
    2       Tristan Stubbs              85.14            188.11                 4.90                   57.34                 13.0
    3         David Miller              83.49            187.42                 7.28                   38.61                 19.0
    4        Shubham Dubey              77.59            201.56                 6.25                   43.33                  9.0
    5       Dinesh Karthik              77.12            184.18                 5.06                   36.95                 11.0
    6        Rahul Tewatia              73.58            150.00                 6.36                   28.74                 28.0
    7      Ravindra Jadeja              72.17

In [30]:
for min_balls in [50]:

    test_pool = pressure_comparison[
        (pressure_comparison["balls_faced_normal"] >= min_balls)
        & (pressure_comparison["balls_faced_high"] >= min_balls)
        & (pressure_comparison["matches_played_high"] >= 5)
    ].copy()

    if test_pool.empty:
        print(f"\nNo eligible batters at {min_balls} balls.")
        continue

    # Calculate performance changes
    test_pool["strike_rate_change_pct"] = (
        (
            test_pool["strike_rate_high"]
            - test_pool["strike_rate_normal"]
        )
        / test_pool["strike_rate_normal"].replace(0, np.nan)
    ) * 100

    test_pool["scoring_score"] = (
        test_pool["strike_rate_high"].rank(pct=True) * 100
    )

    test_pool["retention_score"] = (
        test_pool["strike_rate_change_pct"].rank(pct=True) * 100
    )

    test_pool["survival_score"] = (
        test_pool["dismissal_rate_high"].rank(
            ascending=False, pct=True
        ) * 100
    )

    test_pool["experience_score"] = (
        test_pool["matches_played_high"].rank(pct=True) * 100
    )

    test_pool["reliability_score"] = test_pool[
        [
            "scoring_score",
            "retention_score",
            "survival_score",
            "experience_score"
        ]
    ].mean(axis=1)

    test_pool = test_pool.sort_values(
        "reliability_score", ascending=False
    )

    print(f"\nMinimum {min_balls} balls in BOTH groups")
    print("Eligible batters:", len(test_pool))

    print(
        test_pool[
            ["striker_name", "reliability_score"]
        ].head(10).round(2).to_string(index=False)
    )


Minimum 50 balls in BOTH groups
Eligible batters: 37
        striker_name  reliability_score
         Rinku Singh              90.88
        David Miller              84.46
      Tristan Stubbs              82.77
       Rahul Tewatia              74.66
     Ravindra Jadeja              71.62
         Dhruv Jurel              71.28
Namboori Tilak Varma              70.95
           Tim David              69.26
     Ashutosh Sharma              67.91
          Naman Dhir              64.86


In [34]:
weight_scenarios = {
    "Equal Weight": {
        "scoring_score": 0.25,
        "retention_score": 0.25,
        "survival_score": 0.25,
        "experience_score": 0.25
    },
    "Scoring Focused": {
        "scoring_score": 0.40,
        "retention_score": 0.20,
        "survival_score": 0.25,
        "experience_score": 0.15
    },
    "Survival Focused": {
        "scoring_score": 0.20,
        "retention_score": 0.15,
        "survival_score": 0.45,
        "experience_score": 0.20
    }
}

weight_results = []

for scenario, weights in weight_scenarios.items():

    temp = score_data.copy()

    temp["scenario_score"] = sum(
        temp[column] * weight
        for column, weight in weights.items()
    )

    temp = temp.sort_values(
        "scenario_score", ascending=False
    ).head(10)

    for rank, (_, player) in enumerate(temp.iterrows(), start=1):
        weight_results.append({
            "scenario": scenario,
            "rank": rank,
            "striker_name": player["striker_name"],
            "score": player["scenario_score"]
        })

weight_results_df = pd.DataFrame(weight_results)

print(weight_results_df.round(2).to_string(index=False))


        scenario  rank         striker_name  score
    Equal Weight     1          Rinku Singh  91.27
    Equal Weight     2       Tristan Stubbs  85.14
    Equal Weight     3         David Miller  83.49
    Equal Weight     4        Shubham Dubey  77.59
    Equal Weight     5       Dinesh Karthik  77.12
    Equal Weight     6        Rahul Tewatia  73.58
    Equal Weight     7      Ravindra Jadeja  72.17
    Equal Weight     8          Dhruv Jurel  71.70
    Equal Weight     9 Namboori Tilak Varma  71.70
    Equal Weight    10            Tim David  69.58
 Scoring Focused     1          Rinku Singh  92.97
 Scoring Focused     2       Tristan Stubbs  87.59
 Scoring Focused     3        Shubham Dubey  83.35
 Scoring Focused     4         David Miller  82.83
 Scoring Focused     5       Dinesh Karthik  79.58
 Scoring Focused     6          Dhruv Jurel  72.45
 Scoring Focused     7           Naman Dhir  72.31
 Scoring Focused     8      Ashutosh Sharma  70.71
 Scoring Focused     9 Namboori

In [36]:
print("Available match columns:")
print(matches.columns.tolist())

print("\nFirst 5 match records:")
print(matches.head().to_string(index=False))

Available match columns:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']

First 5 match records:
 match_id  comp_id  match_number match_date  stage  venue_id venue_country  home_team_id   home_team  away_team_id away_team  home_team_captain_id  away_team_captain_id  toss_win_team_id toss_opted  man_of_the_match  first_inn_revised_overs  second_inn_revised_overs  revised_target  knock_out  win_team_id                       result
    13511     1331             1 2022-01-22 League        54   West Indies            28 West Indies            22   England                   NaN                   NaN              28.0   Fielding                 0                      NaN            

In [37]:
competitions = pd.read_csv(DATA_PATH + "competitions_cleaned.csv")

print("Competition columns:")
print(competitions.columns.tolist())

print("\nCompetition sample:")
print(competitions.head(10).to_string(index=False))

Competition columns:
['comp_id', 'comp_name', 'match_format', 'season']

Competition sample:
 comp_id                                      comp_name match_format season
    3395   South Africa in New Zealand T20I Series 2026         T20I   2026
    3405       Afghanistan v Sri Lanka T20I Series 2026         T20I   2026
    3465             India in Zimbabwe T20I Series 2026         T20I   2026
    3568        Bangladesh in Zimbabwe T20I Series 2026         T20I   2026
    3482       Australia in Bangladesh T20I Series 2026         T20I   2026
    3464      Sri Lanka in West Indies T20I Series 2026         T20I   2026
    3765         Sri Lanka in Pakistan T20I Series 2026         T20I   2026
    3739                 India in Japan T20I Match 2026         T20I   2026
    3787        Afghanistan v Zimbabwe T20I Series 2026         T20I   2026
    3713 Afghanistan vs India in India T20I Series 2026         T20I   2026


In [38]:
t20_competitions = competitions[
    competitions["match_format"].str.contains(
        "T20", case=False, na=False
    )
].copy()

t20_competitions = t20_competitions.sort_values(
    ["comp_name", "season"]
)

print("Total T20 competitions:", len(t20_competitions))

print(
    t20_competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ].to_string(index=False)
)

Total T20 competitions: 141
 comp_id                                               comp_name match_format  season
    1885              Afghanistan in Bangladesh T20I Series 2023         T20I    2023
    2059                   Afghanistan in India T20I Series 2024         T20I    2024
    1543                 Afghanistan in Ireland T20I Series 2022         T20I    2022
    2267               Afghanistan in Sri Lanka T20I Series 2024         T20I    2024
    1458                Afghanistan in Zimbabwe T20I Series 2022         T20I    2022
    2753                Afghanistan in Zimbabwe T20I Series 2024         T20I    2024
    3254                Afghanistan in Zimbabwe T20I Series 2025         T20I    2025
    3228               Afghanistan v Bangladesh T20I Series 2025         T20I    2025
    2269                  Afghanistan v Ireland T20I Series 2024         T20I    2024
    1792                 Afghanistan v Pakistan T20I Series 2023         T20I    2023
    3405                Af

In [39]:
print("Competition count by format:")
print(competitions["match_format"].value_counts())

print("\nCompetition count by season and format:")
print(
    competitions.groupby(
        ["season", "match_format"]
    ).size().unstack(fill_value=0)
)

print("\nIPL competitions:")
ipl_comps = competitions[
    competitions["comp_name"].str.contains(
        "Indian Premier League|IPL",
        case=False,
        na=False
    )
]

print(
    ipl_comps[
        ["comp_id", "comp_name", "match_format", "season"]
    ].sort_values("season").to_string(index=False)
)

print("\nIPL competition count:", len(ipl_comps))
print("T20I competition count:",
      (competitions["match_format"] == "T20I").sum())

Competition count by format:
match_format
T20I    134
T20       7
Name: count, dtype: int64

Competition count by season and format:
match_format  T20  T20I
season                 
2022            1    24
2022/23         0     3
2023            1    19
2023/24         0     1
2024            2    31
2024/25         0     1
2025            2    31
2026            1    24

IPL competitions:
 comp_id comp_name match_format season
    1358  IPL 2022          T20   2022
    1715  IPL 2023          T20   2023
    2193  IPL 2024          T20   2024
    2725  IPL 2025          T20   2025
    3298  IPL 2026          T20   2026

IPL competition count: 5
T20I competition count: 134


In [40]:
# Identify IPL competitions
ipl_competitions = competitions[
    competitions["comp_name"].str.contains(
        r"\bIPL\b",
        case=False,
        na=False
    )
]

# Identify T20I competitions
t20i_competitions = competitions[
    competitions["match_format"] == "T20I"
]

# Combine competition IDs
target_comp_ids = pd.concat([
    ipl_competitions["comp_id"],
    t20i_competitions["comp_id"]
]).unique()

# Filter matches
target_matches = matches[
    matches["comp_id"].isin(target_comp_ids)
].copy()

# Attach competition information
target_matches = target_matches.merge(
    competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

print("Target competitions:", len(target_comp_ids))
print("Target matches:", len(target_matches))

print("\nMatches by format:")
print(target_matches["match_format"].value_counts())

print("\nMatches by season and format:")
print(
    target_matches.groupby(
        ["season", "match_format"]
    ).size().unstack(fill_value=0)
)

print("\nSample:")
print(
    target_matches[
        ["match_id", "comp_name", "match_format", "season"]
    ].head(10).to_string(index=False)
)

Target competitions: 139
Target matches: 853

Matches by format:
match_format
T20I    482
T20     371
Name: count, dtype: int64

Matches by season and format:
match_format  T20  T20I
season                 
2022           74    94
2022/23         0    12
2023           74    63
2023/24         0     3
2024           74   109
2024/25         0     3
2025           75   126
2026           74    72

Sample:
 match_id                               comp_name match_format season
    13511 England in West Indies T20I Series 2022         T20I   2022
    13512 England in West Indies T20I Series 2022         T20I   2022
    13513 England in West Indies T20I Series 2022         T20I   2022
    13514 England in West Indies T20I Series 2022         T20I   2022
    13515 England in West Indies T20I Series 2022         T20I   2022
    13522   West Indies in India T20I Series 2022         T20I   2022
    13523   West Indies in India T20I Series 2022         T20I   2022
    13524   West Indies in India

In [41]:
# Filter ball-by-ball records to target matches
target_ball_by_ball = ball_by_ball[
    ball_by_ball["match_id"].isin(target_matches["match_id"])
].copy()

# Compare match coverage
target_match_ids = set(target_matches["match_id"])
ball_match_ids = set(target_ball_by_ball["match_id"])

missing_match_ids = target_match_ids - ball_match_ids
matched_match_ids = target_match_ids & ball_match_ids

print("Target matches:", len(target_match_ids))
print("Matches with ball-by-ball data:", len(matched_match_ids))
print("Matches without ball-by-ball data:", len(missing_match_ids))

print("\nTarget ball-by-ball shape:", target_ball_by_ball.shape)

# Validate coverage by format
coverage = target_ball_by_ball.merge(
    target_matches[["match_id", "match_format"]],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("\nBall-by-ball records by format:")
print(coverage["match_format"].value_counts())

print("\nMissing competition information:",
      coverage["match_format"].isna().sum())

Target matches: 853
Matches with ball-by-ball data: 837
Matches without ball-by-ball data: 16

Target ball-by-ball shape: (194014, 33)

Ball-by-ball records by format:
match_format
T20I    106327
T20      87687
Name: count, dtype: int64

Missing competition information: 0


In [42]:
missing_matches = target_matches[
    target_matches["match_id"].isin(missing_match_ids)
].copy()

print("Matches without ball-by-ball data:", len(missing_matches))

print(
    missing_matches[
        [
            "match_id",
            "comp_name",
            "match_format",
            "season",
            "match_date",
            "result",
            "win_team_id"
        ]
    ].sort_values(
        ["match_format", "match_date"]
    ).to_string(index=False)
)

print("\nMissing matches by format:")
print(missing_matches["match_format"].value_counts())

Matches without ball-by-ball data: 16
 match_id                                  comp_name match_format season match_date          result  win_team_id
    31281                                   IPL 2024          T20   2024 2024-05-13 Match Abandoned          NaN
    31284                                   IPL 2024          T20   2024 2024-05-16 Match Abandoned          NaN
    31288                                   IPL 2024          T20   2024 2024-05-19 Match Abandoned          NaN
    40761                                   IPL 2025          T20   2025 2025-05-17 Match Abandoned          NaN
    32622    Netherlands Tri-Nation T20I Series 2024         T20I   2024 2024-05-20 Match Abandoned          NaN
    32610       Pakistan in England T20I Series 2024         T20I   2024 2024-05-22 Match Abandoned          NaN
    32612       Pakistan in England T20I Series 2024         T20I   2024 2024-05-28 Match Abandoned          NaN
    36564      Australia in England T20I Series 2024      

In [43]:
combined_deliveries = target_ball_by_ball.merge(
    target_matches[
        ["match_id", "comp_id", "comp_name", "match_format", "season"]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("Combined delivery dataset:", combined_deliveries.shape)

print("\nUnique matches:")
print(combined_deliveries["match_id"].nunique())

print("\nDeliveries by format:")
print(
    combined_deliveries.groupby("match_format").size()
)

print("\nDeliveries by season and format:")
print(
    combined_deliveries.groupby(
        ["season", "match_format"]
    ).size().unstack(fill_value=0)
)

print("\nMissing competition metadata:")
print(
    combined_deliveries[
        ["comp_id", "comp_name", "match_format", "season"]
    ].isna().sum()
)

Combined delivery dataset: (194014, 37)

Unique matches:
837

Deliveries by format:
match_format
T20      87687
T20I    106327
dtype: int64

Deliveries by season and format:
match_format    T20   T20I
season                    
2022          17909  21718
2022/23           0   2838
2023          17863  14386
2023/24           0    739
2024          17103  23672
2024/25           0    754
2025          17285  26661
2026          17527  15559

Missing competition metadata:
comp_id         0
comp_name       0
match_format    0
season          0
dtype: int64


In [44]:
print("Delivery records by innings:")
print(
    combined_deliveries.groupby(
        ["match_format", "innings"]
    ).size().unstack(fill_value=0)
)

print("\nUnique matches by innings:")
print(
    combined_deliveries.groupby(
        ["match_format", "innings"]
    )["match_id"].nunique().unstack(fill_value=0)
)

print("\nInnings values:")
print(sorted(combined_deliveries["innings"].dropna().unique()))

Delivery records by innings:
innings           1      2   3  4
match_format                     
T20           45588  42085   9  5
T20I          56466  49849  10  2

Unique matches by innings:
innings         1    2  3  4
match_format                
T20           367  363  2  2
T20I          470  452  2  2

Innings values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [45]:
extra_innings = combined_deliveries[
    combined_deliveries["innings"].isin([3, 4])
].copy()

extra_innings_summary = (
    extra_innings[
        [
            "match_id",
            "comp_name",
            "match_format",
            "season",
            "innings"
        ]
    ]
    .drop_duplicates()
    .sort_values(["match_format", "match_id", "innings"])
)

print("Matches and innings containing innings 3 or 4:")
display(extra_innings_summary)

print("\nNumber of unique matches:")
print(extra_innings_summary["match_id"].nunique())

Matches and innings containing innings 3 or 4:


,match_id,comp_name,match_format,season,innings
128239,40725,IPL 2025,T20,2025,3
128245,40725,IPL 2025,T20,2025,4
178262,50359,IPL 2026,T20,2026,3
178265,50359,IPL 2026,T20,2026,4
105762,34429,India in Sri Lanka T20I Series 2024,T20I,2024,3
105766,34429,India in Sri Lanka T20I Series 2024,T20I,2024,4
151793,45512,Asia Cup 2025,T20I,2025,3
151799,45512,Asia Cup 2025,T20I,2025,4



Number of unique matches:
4


In [46]:
regular_deliveries = combined_deliveries[
    combined_deliveries["innings"].isin([1, 2])
].copy()

print("Regular innings dataset:", regular_deliveries.shape)

print("\nRecords by format:")
print(
    regular_deliveries.groupby("match_format").size()
)

print("\nRecords by innings and format:")
print(
    regular_deliveries.groupby(
        ["match_format", "innings"]
    ).size().unstack(fill_value=0)
)

print("\nUnique matches:")
print(regular_deliveries["match_id"].nunique())

print("\nUnexpected innings values:")
print(
    sorted(regular_deliveries["innings"].unique())
)

Regular innings dataset: (193988, 37)

Records by format:
match_format
T20      87673
T20I    106315
dtype: int64

Records by innings and format:
innings           1      2
match_format              
T20           45588  42085
T20I          56466  49849

Unique matches:
837

Unexpected innings values:
[np.int64(1), np.int64(2)]


In [47]:
print("Extras types:")
print(
    regular_deliveries["extras_type"]
    .value_counts(dropna=False)
)

print("\nDismissal types:")
print(
    regular_deliveries["dismissal_type"]
    .value_counts(dropna=False)
)

print("\nWicket flag:")
print(
    regular_deliveries["is_wicket"]
    .value_counts(dropna=False)
)

print("\nWicket records by dismissal type:")
print(
    regular_deliveries[
        regular_deliveries["is_wicket"] == True
    ]["dismissal_type"]
    .value_counts(dropna=False)
)

print("\nExtras by format:")
print(
    regular_deliveries.groupby(
        ["match_format", "extras_type"],
        dropna=False
    ).size().unstack(fill_value=0)
)

Extras types:
extras_type
NaN         182525
wide          7472
legbyes       2661
Overstep       565
byes           505
other          259
Penalty          1
Name: count, dtype: int64

Dismissal types:
dismissal_type
NaN                      183510
Caught                     7327
Bowled                     1689
LBW                         666
RunOut                      571
Stump                       200
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1
Name: count, dtype: int64

Wicket flag:
is_wicket
False    183510
True      10478
Name: count, dtype: int64

Wicket records by dismissal type:
dismissal_type
Caught                   7327
Bowled                   1689
LBW                       666
RunOut                    571
Stump                     200
Hit Wicket                 17
Retire Out                  6
Obstracting the field       1
Obstructing the field       1
Name: count, dtype: int64

Extras 

In [48]:
wicket_records = regular_deliveries[
    regular_deliveries["is_wicket"] == True
].copy()

print("Wicket records by format:")
print(
    wicket_records.groupby(
        ["match_format", "dismissal_type"]
    ).size().unstack(fill_value=0)
)

print("\nWicket records with missing dismissed_player_id:")
print(
    wicket_records["dismissed_player_id"].isna().sum()
)

print("\nWicket records with missing dismissal_type:")
print(
    wicket_records["dismissal_type"].isna().sum()
)

print("\nDismissal type present but is_wicket is False:")
print(
    regular_deliveries[
        (regular_deliveries["is_wicket"] == False) &
        (regular_deliveries["dismissal_type"].notna())
    ][["match_format", "dismissal_type"]]
    .value_counts()
)

Wicket records by format:
dismissal_type  Bowled  Caught  Hit Wicket  LBW  Obstracting the field  \
match_format                                                             
T20                638    3220           7  255                      1   
T20I              1051    4107          10  411                      0   

dismissal_type  Obstructing the field  Retire Out  RunOut  Stump  
match_format                                                      
T20                                 1           5     246     81  
T20I                                0           1     325    119  

Wicket records with missing dismissed_player_id:
1

Wicket records with missing dismissal_type:
0

Dismissal type present but is_wicket is False:
Series([], Name: count, dtype: int64)


In [49]:
missing_dismissed_player = wicket_records[
    wicket_records["dismissed_player_id"].isna()
]

display(
    missing_dismissed_player[
        [
            "match_id",
            "match_format",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "non_striker_name",
            "bowler_name",
            "dismissal_type",
            "is_wicket"
        ]
    ]
)

,match_id,match_format,innings,overid,over_ball_num,striker_name,non_striker_name,bowler_name,dismissal_type,is_wicket
124380,40709,T20,2,19,5,Hardik Pandya,Namboori Tilak Varma,Shardul Thakur,Retire Out,True


In [50]:
# Dismissal types credited to the bowler
bowler_credited_dismissals = [
    "Caught",
    "Bowled",
    "LBW",
    "Stump",
    "Hit Wicket"
]

regular_deliveries["is_bowler_credited_wicket"] = (
    regular_deliveries["is_wicket"].eq(True) &
    regular_deliveries["dismissal_type"].isin(
        bowler_credited_dismissals
    )
)

# All wicket events count towards wickets lost in the innings
regular_deliveries["is_match_state_wicket"] = (
    regular_deliveries["is_wicket"].eq(True)
)

print("Match-state wicket events:")
print(regular_deliveries["is_match_state_wicket"].sum())

print("\nBowler-credited wicket events:")
print(regular_deliveries["is_bowler_credited_wicket"].sum())

print("\nWicket events excluded from bowler credit:")
print(
    regular_deliveries[
        regular_deliveries["is_match_state_wicket"] &
        ~regular_deliveries["is_bowler_credited_wicket"]
    ]["dismissal_type"].value_counts(dropna=False)
)

Match-state wicket events:
10478

Bowler-credited wicket events:
9899

Wicket events excluded from bowler credit:
dismissal_type
RunOut                   571
Retire Out                 6
Obstracting the field      1
Obstructing the field      1
Name: count, dtype: int64


In [51]:
unusual_extras = regular_deliveries[
    regular_deliveries["extras_type"].isin(["other", "Penalty"])
].copy()

display(
    unusual_extras[
        [
            "match_id",
            "match_format",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

print("\nUnusual extras by type and format:")
print(
    unusual_extras.groupby(
        ["match_format", "extras_type"],
        dropna=False
    ).size()
)

,match_id,match_format,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,T20I,1,10,1,Rovman Powell,Tymal Mills,0,other,1,0,False,NaN
1079,13515,T20I,1,16,6,Rovman Powell,Saqib Mahmood,0,other,1,0,False,NaN
1226,13515,T20I,2,19,5,Sam Billings,Sheldon Cottrell,2,other,1,0,False,NaN
2588,13614,T20I,1,18,5,Dasun Shanaka,Mitchell Starc,0,other,1,4,False,NaN
2609,13614,T20I,2,2,1,Ashton Agar,Nuwan Thushara,0,other,1,0,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
190831,54210,T20I,1,16,4,Bevon Jacobs,Bhuvneshwar Kumar,0,other,1,0,False,NaN
190963,54210,T20I,2,16,6,Shreyas Iyer,Nathan Smith,0,other,1,0,False,NaN
191549,54213,T20I,1,16,2,Glenn Phillips,Nitish Reddy,4,other,1,0,False,NaN
192770,55078,T20I,1,20,4,Eshan Malinga,Razaullah Razaullah,0,other,1,1,False,NaN



Unusual extras by type and format:
match_format  extras_type
T20           other          122
T20I          Penalty          1
              other          137
dtype: int64


In [52]:
penalty_record = regular_deliveries[
    regular_deliveries["extras_type"] == "Penalty"
]

display(
    penalty_record[
        [
            "match_id",
            "comp_name",
            "match_format",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "extras_type"
        ]
    ]
)

,match_id,comp_name,match_format,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type
34203,16753,Afghanistan in Ireland T20I Series 2022,T20I,2,6,4,Harry Tector,Fareed Malik,0,5,0,Penalty


In [53]:
# Total runs added to the team's innings on each delivery
regular_deliveries["total_delivery_runs"] = (
    regular_deliveries["runs"].fillna(0)
    + regular_deliveries["extras1_runs"].fillna(0)
    + regular_deliveries["extras_2_runs"].fillna(0)
)

# Legal delivery: wides and no-balls (Overstep) are excluded
regular_deliveries["is_legal_ball"] = (
    ~regular_deliveries["extras_type"].isin(["wide", "Overstep"])
)

# Batter faces a ball on every delivery except a wide
regular_deliveries["is_ball_faced"] = (
    regular_deliveries["extras_type"] != "wide"
)

# Keep unusual penalty records identifiable
regular_deliveries["is_penalty_delivery"] = (
    regular_deliveries["extras_type"] == "Penalty"
)

print("Total delivery runs:", regular_deliveries["total_delivery_runs"].sum())

print("\nLegal balls:", regular_deliveries["is_legal_ball"].sum())

print("Batter balls faced:", regular_deliveries["is_ball_faced"].sum())

print("Penalty records:", regular_deliveries["is_penalty_delivery"].sum())

print("\nLegal-ball counts by format:")
print(
    regular_deliveries.groupby("match_format")["is_legal_ball"].sum()
)

Total delivery runs: 275335

Legal balls: 185951
Batter balls faced: 186516
Penalty records: 1

Legal-ball counts by format:
match_format
T20      83864
T20I    102087
Name: is_legal_ball, dtype: int64


In [54]:
# Sort deliveries in actual innings order
regular_deliveries = regular_deliveries.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

# Group each match innings separately
innings_groups = regular_deliveries.groupby(
    ["match_id", "innings"],
    sort=False
)

# Cumulative team runs
regular_deliveries["cumulative_runs"] = (
    innings_groups["total_delivery_runs"].cumsum()
)

# Cumulative legal balls
regular_deliveries["cumulative_legal_balls"] = (
    innings_groups["is_legal_ball"].cumsum()
)

# Cumulative wickets lost
regular_deliveries["cumulative_wickets"] = (
    innings_groups["is_match_state_wicket"].cumsum()
)

# Wickets remaining
regular_deliveries["wickets_remaining"] = (
    10 - regular_deliveries["cumulative_wickets"]
)

print("Cumulative state calculated.")

print("\nMissing values:")
print(
    regular_deliveries[
        [
            "cumulative_runs",
            "cumulative_legal_balls",
            "cumulative_wickets",
            "wickets_remaining"
        ]
    ].isna().sum()
)

print("\nWickets remaining summary:")
print(regular_deliveries["wickets_remaining"].describe())

print("\nNegative wickets remaining:")
print((regular_deliveries["wickets_remaining"] < 0).sum())

print("\nSample innings state:")
display(
    regular_deliveries[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "total_delivery_runs",
            "cumulative_runs",
            "cumulative_legal_balls",
            "cumulative_wickets",
            "wickets_remaining"
        ]
    ].head(10)
)

Cumulative state calculated.

Missing values:
cumulative_runs           0
cumulative_legal_balls    0
cumulative_wickets        0
wickets_remaining         0
dtype: int64

Wickets remaining summary:
count    193988.000000
mean          7.267877
std           2.242592
min           0.000000
25%           6.000000
50%           8.000000
75%           9.000000
max          10.000000
Name: wickets_remaining, dtype: float64

Negative wickets remaining:
0

Sample innings state:


,match_id,innings,overid,over_ball_num,total_delivery_runs,cumulative_runs,cumulative_legal_balls,cumulative_wickets,wickets_remaining
0,13511,1,1,1,0,0,1,0,10
1,13511,1,1,2,0,0,2,0,10
2,13511,1,1,3,0,0,3,0,10
3,13511,1,1,4,0,0,4,0,10
4,13511,1,1,5,6,6,5,0,10
5,13511,1,1,6,0,6,6,1,9
6,13511,1,2,1,0,6,7,1,9
7,13511,1,2,2,0,6,8,1,9
8,13511,1,2,3,0,6,9,1,9
9,13511,1,2,4,4,10,10,1,9


In [55]:
innings_summary = (
    regular_deliveries
    .groupby(["match_id", "innings"], as_index=False)
    .tail(1)
    [
        [
            "match_id",
            "innings",
            "cumulative_runs",
            "cumulative_legal_balls",
            "cumulative_wickets"
        ]
    ]
    .rename(columns={
        "cumulative_runs": "innings_total_runs",
        "cumulative_legal_balls": "innings_legal_balls",
        "cumulative_wickets": "innings_total_wickets"
    })
    .sort_values(["match_id", "innings"])
    .reset_index(drop=True)
)

print("Innings summary shape:", innings_summary.shape)

print("\nInnings summaries by format:")
innings_summary_with_format = innings_summary.merge(
    target_matches[["match_id", "match_format"]],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print(
    innings_summary_with_format.groupby(
        ["match_format", "innings"]
    ).size().unstack(fill_value=0)
)

print("\nInnings score summary:")
print(innings_summary["innings_total_runs"].describe())

print("\nSample innings summaries:")
display(innings_summary.head(10))

Innings summary shape: (1652, 5)

Innings summaries by format:
innings         1    2
match_format          
T20           367  363
T20I          470  452

Innings score summary:
count    1652.000000
mean      166.667676
std        39.893351
min         2.000000
25%       143.000000
50%       167.000000
75%       193.000000
max       304.000000
Name: innings_total_runs, dtype: float64

Sample innings summaries:


,match_id,innings,innings_total_runs,innings_legal_balls,innings_total_wickets
0,13511,1,103,118,10
1,13511,2,104,103,1
2,13512,1,171,120,8
3,13512,2,170,120,8
4,13513,1,224,121,5
5,13513,2,204,120,9
6,13514,1,193,120,6
7,13514,2,159,120,5
8,13515,1,179,121,4
9,13515,2,162,120,10


In [56]:
# Attach match-level revised conditions
innings_limit_audit = innings_summary.merge(
    target_matches[
        [
            "match_id",
            "match_format",
            "comp_name",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Expected legal-ball limit for each innings
innings_limit_audit["expected_ball_limit"] = np.where(
    innings_limit_audit["innings"] == 1,
    innings_limit_audit["first_inn_revised_overs"].fillna(120),
    innings_limit_audit["second_inn_revised_overs"].fillna(120)
)

print("Innings with more than 120 legal balls:")
display(
    innings_limit_audit[
        innings_limit_audit["innings_legal_balls"] > 120
    ][
        [
            "match_id",
            "match_format",
            "comp_name",
            "innings",
            "innings_legal_balls",
            "expected_ball_limit",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ]
)

print("\nInnings exceeding their expected ball limit:")
display(
    innings_limit_audit[
        innings_limit_audit["innings_legal_balls"] >
        innings_limit_audit["expected_ball_limit"]
    ][
        [
            "match_id",
            "match_format",
            "innings",
            "innings_legal_balls",
            "expected_ball_limit"
        ]
    ]
)

Innings with more than 120 legal balls:


,match_id,match_format,comp_name,innings,innings_legal_balls,expected_ball_limit,first_inn_revised_overs,second_inn_revised_overs,revised_target,result
4,13513,T20I,England in West Indies T20I Series 2022,1,121,120.0,NaN,NaN,NaN,West Indies Won by 20 runs
8,13515,T20I,England in West Indies T20I Series 2022,1,121,120.0,NaN,NaN,NaN,West Indies Won by 17 runs
20,13614,T20I,Sri Lanka in Australia T20I Series 2022,1,121,120.0,NaN,NaN,NaN,Australia Won by 6 Wkts
26,14094,T20,IPL 2022,1,121,120.0,NaN,NaN,NaN,Kolkata Knight Riders Won by 6 Wkts
35,14098,T20,IPL 2022,2,121,120.0,NaN,NaN,NaN,Rajasthan Royals Won by 61 Runs
...,...,...,...,...,...,...,...,...,...,...
1608,50942,T20,IPL 2026,1,121,120.0,NaN,NaN,NaN,Royal Challengers Bengaluru Won by 92 runs
1610,50943,T20,IPL 2026,1,121,120.0,NaN,NaN,NaN,Rajasthan Royals Won by 47 runs
1622,54210,T20I,India in New Zealand T20I Series 2026,1,121,120.0,NaN,NaN,NaN,India Won by 4 wickets
1628,54213,T20I,India in New Zealand T20I Series 2026,1,121,120.0,NaN,NaN,NaN,New Zealand Won by 87 runs



Innings exceeding their expected ball limit:


,match_id,match_format,innings,innings_legal_balls,expected_ball_limit
4,13513,T20I,1,121,120.0
8,13515,T20I,1,121,120.0
20,13614,T20I,1,121,120.0
26,14094,T20,1,121,120.0
35,14098,T20,2,121,120.0
...,...,...,...,...,...
1608,50942,T20,1,121,120.0
1610,50943,T20,1,121,120.0
1622,54210,T20I,1,121,120.0
1628,54213,T20I,1,121,120.0


In [57]:
# Identify innings that exceed their expected legal-ball limit
excess_innings = innings_limit_audit[
    innings_limit_audit["innings_legal_balls"] >
    innings_limit_audit["expected_ball_limit"]
].copy()

print("Number of affected innings:", len(excess_innings))

# Inspect the final deliveries of the first 5 affected innings
for _, row in excess_innings.head(5).iterrows():

    match_id = row["match_id"]
    innings_no = row["innings"]

    sample = regular_deliveries[
        (regular_deliveries["match_id"] == match_id) &
        (regular_deliveries["innings"] == innings_no)
    ].sort_values(
        ["overid", "over_ball_num"]
    )

    print(
        f"\nMatch: {match_id} | Format: {row['match_format']} "
        f"| Innings: {innings_no}"
    )

    print(
        sample[
            [
                "overid",
                "over_ball_num",
                "extras_type",
                "is_legal_ball",
                "total_delivery_runs",
                "cumulative_legal_balls"
            ]
        ].tail(10).to_string(index=False)
    )

Number of affected innings: 149

Match: 13513 | Format: T20I | Innings: 1
 overid  over_ball_num extras_type  is_legal_ball  total_delivery_runs  cumulative_legal_balls
     19              4         NaN           True                    2                     113
     19              5         NaN           True                    6                     114
     19              6        wide          False                    1                     114
     19              7         NaN           True                    0                     115
     20              1         NaN           True                    0                     116
     20              2         NaN           True                    1                     117
     20              3     legbyes           True                    4                     118
     20              4         NaN           True                    4                     119
     20              5         NaN           True                    4 

In [58]:
# Inspect the over-by-over legal-ball count for the first 5 affected innings

for _, row in excess_innings.head(5).iterrows():

    match_id = row["match_id"]
    innings_no = row["innings"]

    sample = regular_deliveries[
        (regular_deliveries["match_id"] == match_id) &
        (regular_deliveries["innings"] == innings_no)
    ].copy()

    over_summary = (
        sample.groupby("overid")
        .agg(
            delivery_records=("is_legal_ball", "size"),
            legal_balls=("is_legal_ball", "sum"),
            runs=("total_delivery_runs", "sum"),
            final_cumulative_legal_balls=(
                "cumulative_legal_balls", "max"
            )
        )
        .reset_index()
    )

    print(
        f"\nMatch {match_id} | {row['match_format']} "
        f"| Innings {innings_no}"
    )

    display(over_summary)


Match 13513 | T20I | Innings 1


,overid,delivery_records,legal_balls,runs,final_cumulative_legal_balls
0,1,6,6,3,6
1,2,6,6,9,12
2,3,6,6,3,18
3,4,7,6,19,24
4,5,7,6,14,30
5,6,6,6,6,36
6,7,6,6,6,42
7,8,6,6,10,48
8,9,6,6,14,54
9,10,7,7,12,61



Match 13515 | T20I | Innings 1


,overid,delivery_records,legal_balls,runs,final_cumulative_legal_balls
0,1,6,6,3,6
1,2,6,6,19,12
2,3,6,6,9,18
3,4,6,6,2,24
4,5,6,6,14,30
5,6,6,6,11,36
6,7,6,6,2,42
7,8,7,6,9,48
8,9,6,6,9,54
9,10,7,6,8,60



Match 13614 | T20I | Innings 1


,overid,delivery_records,legal_balls,runs,final_cumulative_legal_balls
0,1,8,6,8,6
1,2,6,6,9,12
2,3,6,6,6,18
3,4,6,6,2,24
4,5,7,6,5,30
5,6,6,6,7,36
6,7,6,6,2,42
7,8,6,6,3,48
8,9,6,6,6,54
9,10,6,6,3,60



Match 14094 | T20 | Innings 1


,overid,delivery_records,legal_balls,runs,final_cumulative_legal_balls
0,1,9,6,3,6
1,2,6,6,5,12
2,3,6,6,8,18
3,4,6,6,12,24
4,5,6,6,1,30
5,6,6,6,6,36
6,7,6,6,5,42
7,8,7,6,10,48
8,9,6,6,3,54
9,10,6,6,4,60



Match 14098 | T20 | Innings 2


,overid,delivery_records,legal_balls,runs,final_cumulative_legal_balls
0,1,6,6,2,6
1,2,6,6,1,12
2,3,6,6,4,18
3,4,6,6,0,24
4,5,7,6,6,30
5,6,6,6,1,36
6,7,7,6,7,42
7,8,6,6,7,48
8,9,6,6,4,54
9,10,6,6,4,60


In [59]:
sample_over = regular_deliveries[
    (regular_deliveries["match_id"] == 13513) &
    (regular_deliveries["innings"] == 1) &
    (regular_deliveries["overid"] == 10)
].copy()

display(
    sample_over[
        [
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_legal_ball",
            "total_delivery_runs",
            "is_wicket"
        ]
    ]
)

,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_legal_ball,total_delivery_runs,is_wicket
532,10,1,Rovman Powell,Tymal Mills,0,other,1,0,True,1,False
533,10,2,Rovman Powell,Tymal Mills,6,NaN,0,0,True,6,False
534,10,3,Rovman Powell,Tymal Mills,1,NaN,0,0,True,1,False
535,10,4,Nicholas Pooran,Tymal Mills,2,NaN,0,0,True,2,False
536,10,5,Nicholas Pooran,Tymal Mills,1,NaN,0,0,True,1,False
537,10,6,Rovman Powell,Tymal Mills,0,NaN,0,0,True,0,False
538,10,7,Rovman Powell,Tymal Mills,1,NaN,0,0,True,1,False


In [60]:
over_audit = (
    regular_deliveries
    .groupby(["match_id", "innings", "overid"])
    .agg(
        delivery_records=("is_legal_ball", "size"),
        legal_balls=("is_legal_ball", "sum"),
        extras=("extras_type", lambda x: x.notna().sum())
    )
    .reset_index()
)

print("Total overs:", len(over_audit))

print("\nOvers with more than 6 legal balls:")
print(
    (over_audit["legal_balls"] > 6).sum()
)

print("\nOvers with more than 6 delivery records:")
print(
    (over_audit["delivery_records"] > 6).sum()
)

print("\nExcess legal-ball counts:")
print(
    over_audit["legal_balls"].value_counts().sort_index()
)

print("\nExamples of overs with more than 6 legal balls:")
display(
    over_audit[
        over_audit["legal_balls"] > 6
    ].head(20)
)

Total overs: 31209

Overs with more than 6 legal balls:
240

Overs with more than 6 delivery records:
6500

Excess legal-ball counts:
legal_balls
0        1
1       92
2       97
3      101
4      130
5      136
6    30412
7      231
8        8
9        1
Name: count, dtype: int64

Examples of overs with more than 6 legal balls:


,match_id,innings,overid,delivery_records,legal_balls,extras
87,13513,1,10,7,7,1
173,13515,1,16,8,7,2
196,13515,2,19,7,7,2
413,13614,1,18,7,7,1
417,13614,2,2,7,7,3
530,14094,1,19,8,7,2
708,14098,2,20,7,7,1
804,14101,1,17,7,7,1
860,14102,2,19,10,7,4
1055,14107,2,16,7,7,1


In [61]:
# Get all delivery records belonging to overs with more than 6 legal balls
anomalous_overs = over_audit[
    over_audit["legal_balls"] > 6
][["match_id", "innings", "overid"]]

anomalous_deliveries = regular_deliveries.merge(
    anomalous_overs,
    on=["match_id", "innings", "overid"],
    how="inner"
)

print("Anomalous delivery records:", anomalous_deliveries.shape)

print("\nExtras types in anomalous overs:")
print(
    anomalous_deliveries["extras_type"]
    .value_counts(dropna=False)
)

print("\nExtras types by legal-ball status:")
print(
    anomalous_deliveries.groupby(
        ["extras_type", "is_legal_ball"],
        dropna=False
    ).size().unstack(fill_value=0)
)

print("\nAnomalous overs by format:")
print(
    anomalous_deliveries.groupby("match_format")[
        "match_id"
    ].nunique()
)

Anomalous delivery records: (1817, 47)

Extras types in anomalous overs:
extras_type
NaN         1409
other        249
wide         119
legbyes       24
byes           8
Overstep       8
Name: count, dtype: int64

Extras types by legal-ball status:
is_legal_ball  False  True 
extras_type                
Overstep           8      0
byes               0      8
legbyes            0     24
other              0    249
wide             119      0
NaN                0   1409

Anomalous overs by format:
match_format
T20      97
T20I    106
Name: match_id, dtype: int64


In [62]:
other_anomalies = anomalous_deliveries[
    anomalous_deliveries["extras_type"] == "other"
].copy()

print("Other extras records:", len(other_anomalies))

print("\nOther extras run combinations:")
print(
    other_anomalies.groupby(
        ["extras1_runs", "extras_2_runs"],
        dropna=False
    ).size().sort_values(ascending=False)
)

print("\nSample other extras records:")
display(
    other_anomalies[
        [
            "match_id",
            "match_format",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "extras_type",
            "is_wicket"
        ]
    ].head(20)
)

Other extras records: 249

Other extras run combinations:
extras1_runs  extras_2_runs
1             0                236
              4                  7
              1                  6
dtype: int64

Sample other extras records:


,match_id,match_format,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_wicket
0,13513,T20I,1,10,1,Rovman Powell,Tymal Mills,0,1,0,other,False
12,13515,T20I,1,16,6,Rovman Powell,Saqib Mahmood,0,1,0,other,False
19,13515,T20I,2,19,5,Sam Billings,Sheldon Cottrell,2,1,0,other,False
26,13614,T20I,1,18,5,Dasun Shanaka,Mitchell Starc,0,1,4,other,False
29,13614,T20I,2,2,1,Ashton Agar,Nuwan Thushara,0,1,0,other,False
42,14094,T20,1,19,7,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,other,False
44,14098,T20,2,20,1,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,other,False
54,14101,T20,1,17,4,Kagiso Rabada,Tim Southee,0,1,0,other,False
66,14102,T20,2,19,9,Kieron Pollard,Prasidh Krishna,0,1,0,other,True
72,14107,T20,2,16,5,Patrick Cummins,Daniel Sams,2,1,0,other,False


In [63]:
# Alternative interpretation for diagnostic purposes only
regular_deliveries["legal_ball_alternative"] = (
    ~regular_deliveries["extras_type"].isin(
        ["wide", "Overstep", "other"]
    )
)

alternative_over_audit = (
    regular_deliveries
    .groupby(["match_id", "innings", "overid"])
    .agg(
        original_legal_balls=("is_legal_ball", "sum"),
        alternative_legal_balls=("legal_ball_alternative", "sum")
    )
    .reset_index()
)

print("Original overs with more than 6 legal balls:")
print(
    (alternative_over_audit["original_legal_balls"] > 6).sum()
)

print("\nAlternative overs with more than 6 legal balls:")
print(
    (alternative_over_audit["alternative_legal_balls"] > 6).sum()
)

print("\nAlternative legal-ball distribution:")
print(
    alternative_over_audit["alternative_legal_balls"]
    .value_counts()
    .sort_index()
)

print("\nOvers still exceeding 6 under alternative rule:")
display(
    alternative_over_audit[
        alternative_over_audit["alternative_legal_balls"] > 6
    ].head(15)
)

Original overs with more than 6 legal balls:
240

Alternative overs with more than 6 legal balls:
1

Alternative legal-ball distribution:
alternative_legal_balls
0        1
1       95
2       95
3      101
4      130
5      139
6    30647
7        1
Name: count, dtype: int64

Overs still exceeding 6 under alternative rule:


,match_id,innings,overid,original_legal_balls,alternative_legal_balls
22338,41509,1,12,7,7


In [64]:
remaining_anomaly = alternative_over_audit[
    alternative_over_audit["alternative_legal_balls"] > 6
]

display(remaining_anomaly)

for _, row in remaining_anomaly.iterrows():

    sample = regular_deliveries[
        (regular_deliveries["match_id"] == row["match_id"]) &
        (regular_deliveries["innings"] == row["innings"]) &
        (regular_deliveries["overid"] == row["overid"])
    ]

    display(
        sample[
            [
                "match_id",
                "innings",
                "overid",
                "over_ball_num",
                "striker_name",
                "bowler_name",
                "extras_type",
                "runs",
                "extras1_runs",
                "extras_2_runs",
                "is_legal_ball",
                "legal_ball_alternative"
            ]
        ]
    )

,match_id,innings,overid,original_legal_balls,alternative_legal_balls
22338,41509,1,12,7,7


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,extras_type,runs,extras1_runs,extras_2_runs,is_legal_ball,legal_ball_alternative
138915,41509,1,12,1,Shimron Hetmyer,Aaron Hardie,NaN,0,0,0,True,True
138916,41509,1,12,2,Shimron Hetmyer,Aaron Hardie,NaN,1,0,0,True,True
138917,41509,1,12,3,Jason Holder,Aaron Hardie,NaN,6,0,0,True,True
138918,41509,1,12,4,Jason Holder,Aaron Hardie,NaN,4,0,0,True,True
138919,41509,1,12,5,Jason Holder,Aaron Hardie,NaN,0,0,0,True,True
138920,41509,1,12,6,Jason Holder,Aaron Hardie,NaN,1,0,0,True,True
138921,41509,1,12,7,Shimron Hetmyer,Aaron Hardie,NaN,2,0,0,True,True


In [65]:
print("Match metadata:")
display(
    target_matches[
        target_matches["match_id"] == 41509
    ]
)

print("\nComplete first innings delivery sequence:")
display(
    regular_deliveries[
        (regular_deliveries["match_id"] == 41509) &
        (regular_deliveries["innings"] == 1)
    ][
        [
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "is_wicket",
            "cumulative_runs",
            "cumulative_legal_balls"
        ]
    ].tail(15)
)

print("\nFirst innings summary:")
display(
    innings_summary[
        (innings_summary["match_id"] == 41509) &
        (innings_summary["innings"] == 1)
    ]
)

Match metadata:


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result,comp_name,match_format,season
607,41509,2874,5,2025-07-28,Others,106,West Indies,28,West Indies,20,...,3641,NaN,NaN,NaN,0,20.0,Australia Won by 3 wickets,Australia in West Indies T20I Series 2025,T20I,2025



Complete first innings delivery sequence:


,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,is_wicket,cumulative_runs,cumulative_legal_balls
138954,18,2,Matthew Forde,Nathan Ellis,0,NaN,True,157,106
138955,18,3,Akeal Hosein,Nathan Ellis,1,NaN,False,158,107
138956,18,4,Alzarri Joseph,Nathan Ellis,1,NaN,False,159,108
138957,18,5,Akeal Hosein,Nathan Ellis,0,NaN,False,159,109
138958,18,6,Akeal Hosein,Nathan Ellis,0,NaN,False,159,110
138959,19,1,Alzarri Joseph,Adam Zampa,0,NaN,True,159,111
138960,19,2,Jediah Blades,Adam Zampa,1,NaN,False,160,112
138961,19,3,Akeal Hosein,Adam Zampa,4,NaN,False,164,113
138962,19,4,Akeal Hosein,Adam Zampa,0,NaN,False,164,114
138963,19,5,Akeal Hosein,Adam Zampa,1,NaN,False,165,115



First innings summary:


,match_id,innings,innings_total_runs,innings_legal_balls,innings_total_wickets
1176,41509,1,170,120,10


In [66]:
match_41509_overs = (
    regular_deliveries[
        regular_deliveries["match_id"] == 41509
    ]
    .groupby(["innings", "overid"])
    .agg(
        delivery_records=("is_legal_ball", "size"),
        original_legal_balls=("is_legal_ball", "sum"),
        alternative_legal_balls=("legal_ball_alternative", "sum"),
        over_runs=("total_delivery_runs", "sum")
    )
    .reset_index()
)

display(match_41509_overs)

print("\nLegal balls by innings:")
print(
    match_41509_overs.groupby("innings")[
        ["original_legal_balls", "alternative_legal_balls"]
    ].sum()
)

,innings,overid,delivery_records,original_legal_balls,alternative_legal_balls,over_runs
0,1,1,6,6,6,6
1,1,2,6,6,6,10
2,1,3,6,6,6,5
3,1,4,7,6,6,2
4,1,5,6,6,6,14
5,1,6,6,6,6,12
6,1,7,6,6,6,12
7,1,8,6,6,6,7
8,1,9,6,6,6,7
9,1,10,6,6,6,9



Legal balls by innings:
         original_legal_balls  alternative_legal_balls
innings                                               
1                         120                      119
2                         102                      102


In [67]:
match_41509_sequence = regular_deliveries[
    (regular_deliveries["match_id"] == 41509) &
    (regular_deliveries["innings"] == 1)
].copy()

display(
    match_41509_sequence[
        [
            "innings_ball_num",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "extras_type",
            "is_legal_ball",
            "legal_ball_alternative",
            "is_wicket"
        ]
    ].to_string(index=False)
)

print("\nUnique innings ball numbers:")
print(match_41509_sequence["innings_ball_num"].nunique())

print("\nMissing innings ball numbers:")
expected = set(range(1, 121))
actual = set(match_41509_sequence["innings_ball_num"].dropna().astype(int))
print(sorted(expected - actual))

print("\nDuplicate innings ball numbers:")
print(
    match_41509_sequence[
        match_41509_sequence["innings_ball_num"].duplicated(keep=False)
    ][["innings_ball_num", "overid", "over_ball_num"]]
)

' innings_ball_num  overid  over_ball_num        striker_name   bowler_name extras_type  is_legal_ball  legal_ball_alternative  is_wicket\n                1       1              1        Brandon King  Aaron Hardie         NaN           True                    True      False\n                2       1              2           Shai Hope  Aaron Hardie         NaN           True                    True      False\n                3       1              3           Shai Hope  Aaron Hardie         NaN           True                    True      False\n                4       1              4           Shai Hope  Aaron Hardie         NaN           True                    True      False\n                5       1              5        Brandon King  Aaron Hardie         NaN           True                    True      False\n                6       1              6        Brandon King  Aaron Hardie         NaN           True                    True      False\n                7       2        


Unique innings ball numbers:
121

Missing innings ball numbers:
[]

Duplicate innings ball numbers:
Empty DataFrame
Columns: [innings_ball_num, overid, over_ball_num]
Index: []


In [68]:
# Compare innings legal-ball totals under both interpretations

legal_ball_comparison = (
    regular_deliveries
    .groupby(
        ["match_id", "innings", "match_format"],
        as_index=False
    )
    .agg(
        original_legal_balls=("is_legal_ball", "sum"),
        alternative_legal_balls=("legal_ball_alternative", "sum"),
        delivery_records=("match_id", "size")
    )
)

print("Original legal-ball distribution:")
print(
    legal_ball_comparison["original_legal_balls"]
    .value_counts().sort_index()
)

print("\nAlternative legal-ball distribution:")
print(
    legal_ball_comparison["alternative_legal_balls"]
    .value_counts().sort_index()
)

print("\nInnings with different legal-ball totals:")
print(
    (
        legal_ball_comparison["original_legal_balls"] !=
        legal_ball_comparison["alternative_legal_balls"]
    ).sum()
)

print("\nInnings with more than 120 original legal balls:")
print(
    (
        legal_ball_comparison["original_legal_balls"] > 120
    ).sum()
)

Original legal-ball distribution:
original_legal_balls
2        1
6        1
13       1
21       1
22       2
      ... 
120    837
121    130
122     16
123      1
124      1
Name: count, Length: 85, dtype: int64

Alternative legal-ball distribution:
alternative_legal_balls
2        1
6        1
13       1
21       1
22       2
      ... 
116     47
117     35
118     37
119     60
120    973
Name: count, Length: 81, dtype: int64

Innings with different legal-ball totals:
225

Innings with more than 120 original legal balls:
148


In [69]:
print("Innings with more than 120 original legal balls:")
print(
    legal_ball_comparison[
        legal_ball_comparison["original_legal_balls"] > 120
    ][
        [
            "match_id",
            "innings",
            "match_format",
            "original_legal_balls",
            "alternative_legal_balls",
            "delivery_records"
        ]
    ].head(20).to_string(index=False)
)

Innings with more than 120 original legal balls:
 match_id  innings match_format  original_legal_balls  alternative_legal_balls  delivery_records
    13513        1         T20I                   121                      120               125
    13515        1         T20I                   121                      120               127
    13614        1         T20I                   121                      120               125
    14094        1          T20                   121                      120               126
    14098        2          T20                   121                      120               125
    14102        2          T20                   121                      120               130
    14115        2          T20                   121                      120               129
    14117        1          T20                   121                      120               124
    14127        2          T20                   121                      120

In [70]:
print("Original rule:")
print("Innings above 120:", (legal_ball_comparison["original_legal_balls"] > 120).sum())
print("Innings exactly 120:", (legal_ball_comparison["original_legal_balls"] == 120).sum())
print("Maximum:", legal_ball_comparison["original_legal_balls"].max())

print("\nAlternative rule (excluding 'other'):")
print("Innings above 120:", (legal_ball_comparison["alternative_legal_balls"] > 120).sum())
print("Innings exactly 120:", (legal_ball_comparison["alternative_legal_balls"] == 120).sum())
print("Maximum:", legal_ball_comparison["alternative_legal_balls"].max())

print("\nAlternative rule: innings below 120")
print(
    legal_ball_comparison[
        legal_ball_comparison["alternative_legal_balls"] < 120
    ].shape[0]
)

Original rule:
Innings above 120: 148
Innings exactly 120: 837
Maximum: 124

Alternative rule (excluding 'other'):
Innings above 120: 0
Innings exactly 120: 973
Maximum: 120

Alternative rule: innings below 120
679


In [71]:
other_extras = regular_deliveries[
    regular_deliveries["extras_type"] == "other"
].copy()

print("Other extras by format:")
print(other_extras["match_format"].value_counts())

print("\nOther extras combinations:")
print(
    other_extras[
        ["runs", "extras1_runs", "extras_2_runs"]
    ].value_counts().head(20)
)

print("\nOther extras sample:")
print(
    other_extras[
        [
            "match_id", "match_format", "innings",
            "overid", "over_ball_num", "innings_ball_num",
            "striker_name", "bowler_name",
            "runs", "extras1_runs", "extras_2_runs"
        ]
    ].head(25).to_string(index=False)
)

Other extras by format:
match_format
T20I    137
T20     122
Name: count, dtype: int64

Other extras combinations:
runs  extras1_runs  extras_2_runs
0     1             0                112
1     1             0                 48
4     1             0                 41
6     1             0                 26
2     1             0                 18
0     1             4                  7
                    1                  6
3     1             0                  1
Name: count, dtype: int64

Other extras sample:
 match_id match_format  innings  overid  over_ball_num  innings_ball_num         striker_name         bowler_name  runs  extras1_runs  extras_2_runs
    13513         T20I        1      10              1                57        Rovman Powell         Tymal Mills     0             1              0
    13515         T20I        1      16              6                99        Rovman Powell       Saqib Mahmood     0             1              0
    13515         T20I      

In [72]:
print("Compare extras categories:")

print(
    regular_deliveries[
        regular_deliveries["extras_type"].isin(["other", "Overstep", "wide"])
    ]
    .groupby("extras_type")
    .agg(
        records=("match_id", "size"),
        extra1_values=("extras1_runs", lambda x: sorted(x.dropna().unique().tolist())),
        extra2_values=("extras_2_runs", lambda x: sorted(x.dropna().unique().tolist())),
        batter_run_values=("runs", lambda x: sorted(x.dropna().unique().tolist())),
        legal_count=("is_legal_ball", "sum")
    )
)

print("\nOther vs Overstep examples:")
print(
    regular_deliveries[
        regular_deliveries["extras_type"].isin(["other", "Overstep"])
    ][
        [
            "match_id", "innings", "overid", "over_ball_num",
            "innings_ball_num", "striker_name", "bowler_name",
            "runs", "extras1_runs", "extras_2_runs",
            "extras_type", "delivery_style"
        ]
    ].head(30).to_string(index=False)
)

Compare extras categories:
             records    extra1_values extra2_values   batter_run_values  \
extras_type                                                               
Overstep         565              [1]     [0, 1, 4]  [0, 1, 2, 3, 4, 6]   
other            259              [1]     [0, 1, 4]  [0, 1, 2, 3, 4, 6]   
wide            7472  [1, 2, 3, 4, 5]           [0]                 [0]   

             legal_count  
extras_type               
Overstep               0  
other                259  
wide                   0  

Other vs Overstep examples:
 match_id  innings  overid  over_ball_num  innings_ball_num         striker_name         bowler_name  runs  extras1_runs  extras_2_runs extras_type delivery_style
    13513        1       4              2                20      Nicholas Pooran       George Garton     0             1              0    Overstep        Inswing
    13513        1      10              1                57        Rovman Powell         Tymal Mills     0 

In [73]:
print("Other extras by delivery style:")
print(
    other_extras["delivery_style"]
    .value_counts(dropna=False)
)

print("\nOther extras by format and delivery style:")
print(
    pd.crosstab(
        other_extras["delivery_style"],
        other_extras["match_format"],
        dropna=False
    )
)

Other extras by delivery style:
delivery_style
Beamer               115
Full Toss             70
Bouncer               16
Slow Bouncer           8
Off Cutter             7
Seamed In              6
Back of Hand           4
Leg Cut                3
Off Break              3
Yorker                 3
Leg Break              3
Straight               2
Orthodox               2
Googly                 2
Angled In              2
Knuckle                2
Seamed Out             2
Out Swing              2
NaN                    1
Reverse Swing Out      1
Wide Yorker            1
Arm ball               1
Flipper                1
Carrom                 1
Inswing                1
Name: count, dtype: int64

Other extras by format and delivery style:
match_format       T20  T20I
delivery_style              
Angled In            0     2
Arm ball             0     1
Back of Hand         2     2
Beamer              60    55
Bouncer              8     8
Carrom               1     0
Flipper              1    

In [74]:
regular_deliveries["is_legal_ball_v2"] = (
    ~regular_deliveries["extras_type"].isin(
        ["wide", "Overstep", "other"]
    )
)

legal_ball_v2_summary = (
    regular_deliveries
    .groupby(["match_id", "innings", "match_format"])
    .agg(
        legal_balls=("is_legal_ball_v2", "sum"),
        delivery_records=("match_id", "size")
    )
    .reset_index()
)

print("Revised legal-ball validation:")
print("Total legal balls:", regular_deliveries["is_legal_ball_v2"].sum())
print("Innings above 120:", (legal_ball_v2_summary["legal_balls"] > 120).sum())
print("Innings exactly 120:", (legal_ball_v2_summary["legal_balls"] == 120).sum())
print("Innings below 120:", (legal_ball_v2_summary["legal_balls"] < 120).sum())
print("Maximum legal balls:", legal_ball_v2_summary["legal_balls"].max())

print("\nInnings above 120:")
print(
    legal_ball_v2_summary[
        legal_ball_v2_summary["legal_balls"] > 120
    ].to_string(index=False)
)

Revised legal-ball validation:
Total legal balls: 185692
Innings above 120: 0
Innings exactly 120: 973
Innings below 120: 679
Maximum legal balls: 120

Innings above 120:
Empty DataFrame
Columns: [match_id, innings, match_format, legal_balls, delivery_records]
Index: []


In [76]:
revised_innings = (
    regular_deliveries[
        ["match_id", "innings", "match_format"]
    ]
    .drop_duplicates()
    .merge(
        target_matches[
            [
                "match_id",
                "first_inn_revised_overs",
                "second_inn_revised_overs"
            ]
        ],
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

revised_innings = revised_innings[
    revised_innings[
        ["first_inn_revised_overs", "second_inn_revised_overs"]
    ].notna().any(axis=1)
]

print("Innings with revised overs:")
print(revised_innings.to_string(index=False))

print("\nTotal revised innings:", len(revised_innings))

Innings with revised overs:
 match_id  innings match_format  first_inn_revised_overs  second_inn_revised_overs
    13612        1         T20I                      NaN                     114.0
    13612        2         T20I                      NaN                     114.0
    15410        1         T20I                    114.0                       NaN
    15546        1         T20I                     84.0                       NaN
    16753        1         T20I                     66.0                       NaN
    16753        2         T20I                     66.0                       NaN
    16754        1         T20I                      NaN                      42.0
    16754        2         T20I                      NaN                      42.0
    17395        1         T20I                     48.0                      48.0
    17395        2         T20I                     48.0                      48.0
    17765        1         T20I                     72.0   

In [77]:
innings_limit_check = (
    legal_ball_v2_summary
    .merge(
        target_matches[
            [
                "match_id",
                "first_inn_revised_overs",
                "second_inn_revised_overs"
            ]
        ],
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

innings_limit_check["innings_ball_limit"] = np.where(
    innings_limit_check["innings"] == 1,
    innings_limit_check["first_inn_revised_overs"],
    innings_limit_check["second_inn_revised_overs"]
)

innings_limit_check["innings_ball_limit"] = (
    innings_limit_check["innings_ball_limit"].fillna(120)
)

innings_limit_check["exceeded_limit"] = (
    innings_limit_check["legal_balls"]
    > innings_limit_check["innings_ball_limit"]
)

print("Innings exceeding their applicable ball limit:")
print(
    innings_limit_check[
        innings_limit_check["exceeded_limit"]
    ].to_string(index=False)
)

print("\nTotal innings exceeding limit:",
      innings_limit_check["exceeded_limit"].sum())

print("\nInnings by applicable ball limit:")
print(
    innings_limit_check["innings_ball_limit"]
    .value_counts().sort_index()
)

Innings exceeding their applicable ball limit:
Empty DataFrame
Columns: [match_id, innings, match_format, legal_balls, delivery_records, first_inn_revised_overs, second_inn_revised_overs, innings_ball_limit, exceeded_limit]
Index: []

Total innings exceeding limit: 0

Innings by applicable ball limit:
innings_ball_limit
30.0        4
42.0        3
48.0        4
54.0        4
60.0        3
66.0        4
72.0        2
78.0        2
84.0        4
90.0        7
102.0       4
108.0       2
114.0       5
120.0    1604
Name: count, dtype: int64


In [78]:
# Use the validated provisional legal-ball classification
regular_deliveries["is_legal_ball"] = (
    regular_deliveries["is_legal_ball_v2"]
)

# Recalculate cumulative innings state
regular_deliveries = regular_deliveries.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

innings_groups = regular_deliveries.groupby(
    ["match_id", "innings"], sort=False
)

regular_deliveries["cumulative_legal_balls"] = (
    innings_groups["is_legal_ball"].cumsum()
)

print("Final legal-ball count:",
      regular_deliveries["is_legal_ball"].sum())

print("Missing legal-ball values:",
      regular_deliveries["is_legal_ball"].isna().sum())

print("Maximum legal balls in any innings:",
      regular_deliveries.groupby(
          ["match_id", "innings"]
      )["is_legal_ball"].sum().max())

Final legal-ball count: 185692
Missing legal-ball values: 0
Maximum legal balls in any innings: 120


In [79]:
innings_summary = (
    regular_deliveries
    .groupby(["match_id", "innings"], sort=False)
    .tail(1)
    [
        [
            "match_id",
            "innings",
            "cumulative_runs",
            "cumulative_legal_balls",
            "cumulative_wickets"
        ]
    ]
    .rename(columns={
        "cumulative_runs": "innings_total_runs",
        "cumulative_legal_balls": "innings_legal_balls",
        "cumulative_wickets": "innings_total_wickets"
    })
    .sort_values(["match_id", "innings"])
    .reset_index(drop=True)
)

print("Updated innings summary:", innings_summary.shape)

print("\nLegal balls summary:")
print(innings_summary["innings_legal_balls"].describe())

print("\nInnings by format:")
print(
    innings_summary
    .merge(
        target_matches[["match_id", "match_format"]],
        on="match_id",
        how="left"
    )
    .groupby("match_format")
    .size()
)

Updated innings summary: (1652, 5)

Legal balls summary:
count    1652.000000
mean      112.404358
std        16.260707
min         2.000000
25%       113.000000
50%       120.000000
75%       120.000000
max       120.000000
Name: innings_legal_balls, dtype: float64

Innings by format:
match_format
T20     730
T20I    922
dtype: int64


In [80]:
# Attach match-specific innings ball limits
regular_deliveries = regular_deliveries.merge(
    target_matches[
        [
            "match_id",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Select the correct limit for each innings
regular_deliveries["innings_ball_limit"] = np.where(
    regular_deliveries["innings"] == 1,
    regular_deliveries["first_inn_revised_overs"],
    regular_deliveries["second_inn_revised_overs"]
)

# Use 120 balls when no revised limit exists
regular_deliveries["innings_ball_limit"] = (
    regular_deliveries["innings_ball_limit"].fillna(120)
)

# Calculate remaining legal balls
regular_deliveries["balls_remaining"] = (
    regular_deliveries["innings_ball_limit"]
    - regular_deliveries["cumulative_legal_balls"]
)

print("Missing ball limits:",
      regular_deliveries["innings_ball_limit"].isna().sum())

print("Negative balls remaining:",
      (regular_deliveries["balls_remaining"] < 0).sum())

print("\nBalls remaining summary:")
print(regular_deliveries["balls_remaining"].describe())

print("\nSample:")
print(
    regular_deliveries[
        [
            "match_id", "innings", "innings_ball_limit",
            "cumulative_legal_balls", "balls_remaining"
        ]
    ].head(10).to_string(index=False)
)

Missing ball limits: 0
Negative balls remaining: 0

Balls remaining summary:
count    193988.000000
mean         61.281801
std          33.973310
min           0.000000
25%          32.000000
50%          62.000000
75%          91.000000
max         120.000000
Name: balls_remaining, dtype: float64

Sample:
 match_id  innings  innings_ball_limit  cumulative_legal_balls  balls_remaining
    13511        1               120.0                       1            119.0
    13511        1               120.0                       2            118.0
    13511        1               120.0                       3            117.0
    13511        1               120.0                       4            116.0
    13511        1               120.0                       5            115.0
    13511        1               120.0                       6            114.0
    13511        1               120.0                       7            113.0
    13511        1               120.0              

In [81]:
# Extract first-innings scores
first_innings_scores = (
    innings_summary[
        innings_summary["innings"] == 1
    ][["match_id", "innings_total_runs"]]
    .rename(columns={
        "innings_total_runs": "first_innings_score"
    })
)

# Prepare match-level target information
match_targets = target_matches[
    ["match_id", "revised_target", "match_format", "season"]
].merge(
    first_innings_scores,
    on="match_id",
    how="left",
    validate="one_to_one"
)

# Normal target = first innings score + 1
match_targets["chase_target"] = (
    match_targets["first_innings_score"] + 1
)

# Apply revised target wherever available
match_targets["chase_target"] = (
    match_targets["revised_target"]
    .combine_first(match_targets["chase_target"])
)

print("Match target dataset:", match_targets.shape)

print("\nMissing chase targets:",
      match_targets["chase_target"].isna().sum())

print("\nRevised target matches:",
      match_targets["revised_target"].notna().sum())

print("\nSample:")
print(match_targets.head(10).to_string(index=False))

Match target dataset: (853, 6)

Missing chase targets: 16

Revised target matches: 23

Sample:
 match_id  revised_target match_format season  first_innings_score  chase_target
    13511             NaN         T20I   2022                103.0         104.0
    13512             NaN         T20I   2022                171.0         172.0
    13513             NaN         T20I   2022                224.0         225.0
    13514             NaN         T20I   2022                193.0         194.0
    13515             NaN         T20I   2022                179.0         180.0
    13522             NaN         T20I   2022                157.0         158.0
    13523             NaN         T20I   2022                186.0         187.0
    13524             NaN         T20I   2022                184.0         185.0
    13612           143.0         T20I   2022                149.0         143.0
    13613             NaN         T20I   2022                164.0         165.0


In [82]:
missing_targets = match_targets[
    match_targets["chase_target"].isna()
].merge(
    target_matches[
        ["match_id", "match_date", "comp_name", "result"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

print("Matches with missing chase targets:")
print(
    missing_targets[
        ["match_id", "match_date", "comp_name", "result"]
    ].to_string(index=False)
)

print("\nTotal:", len(missing_targets))

Matches with missing chase targets:
 match_id match_date                                  comp_name          result
    31281 2024-05-13                                   IPL 2024 Match Abandoned
    31284 2024-05-16                                   IPL 2024 Match Abandoned
    31288 2024-05-19                                   IPL 2024 Match Abandoned
    32610 2024-05-22       Pakistan in England T20I Series 2024 Match Abandoned
    32612 2024-05-28       Pakistan in England T20I Series 2024 Match Abandoned
    32622 2024-05-20    Netherlands Tri-Nation T20I Series 2024 Match Abandoned
    38797 2024-12-14  Pakistan in South Africa T20I Series 2024 Match Abandoned
    36564 2024-09-15      Australia in England T20I Series 2024 Match Abandoned
    36210 2025-09-14   South Africa in England T20I Series 2025 Match Abandoned
    40761 2025-05-17                                   IPL 2025 Match Abandoned
    41502 2025-06-12    West Indies in Ireland T20I Series 2025 Match Abandoned
    

In [85]:
regular_deliveries["match_format"] = regular_deliveries["match_format_x"]
regular_deliveries["season"] = regular_deliveries["season_x"]

regular_deliveries = regular_deliveries.drop(
    columns=[
        "match_format_x",
        "match_format_y",
        "season_x",
        "season_y"
    ]
)

print("Updated shape:", regular_deliveries.shape)

print(
    regular_deliveries[
        ["match_id", "innings", "chase_target", "match_format", "season"]
    ].head(10).to_string(index=False)
)

print("\nMissing targets:", regular_deliveries["chase_target"].isna().sum())

Updated shape: (193988, 54)
 match_id  innings  chase_target match_format season
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022
    13511        1         104.0         T20I   2022

Missing targets: 0


In [84]:
print([
    col for col in regular_deliveries.columns
    if "match_format" in col
    or "season" in col
    or "chase_target" in col
])

['match_format_x', 'season_x', 'chase_target', 'match_format_y', 'season_y']


In [86]:
regular_deliveries["runs_remaining"] = (
    regular_deliveries["chase_target"]
    - regular_deliveries["cumulative_runs"]
)

# Required run rate for second innings only
regular_deliveries["required_run_rate"] = np.where(
    (regular_deliveries["innings"] == 2) &
    (regular_deliveries["balls_remaining"] > 0) &
    (regular_deliveries["runs_remaining"] > 0),
    (
        regular_deliveries["runs_remaining"]
        / regular_deliveries["balls_remaining"]
    ) * 6,
    np.nan
)

print("Missing runs remaining:",
      regular_deliveries["runs_remaining"].isna().sum())

print("\nRequired run rate summary:")
print(
    regular_deliveries.loc[
        regular_deliveries["innings"] == 2,
        "required_run_rate"
    ].describe()
)

print("\nSample second-innings deliveries:")
print(
    regular_deliveries.loc[
        regular_deliveries["innings"] == 2,
        [
            "match_id", "cumulative_runs", "chase_target",
            "runs_remaining", "balls_remaining",
            "required_run_rate"
        ]
    ].head(10).to_string(index=False)
)

Missing runs remaining: 0

Required run rate summary:
count    91273.000000
mean        11.492517
std         13.838916
min          0.067416
25%          7.684211
50%          9.681818
75%         12.000000
max        810.000000
Name: required_run_rate, dtype: float64

Sample second-innings deliveries:
 match_id  cumulative_runs  chase_target  runs_remaining  balls_remaining  required_run_rate
    13511                0         104.0           104.0            119.0           5.243697
    13511                4         104.0           100.0            118.0           5.084746
    13511                4         104.0           100.0            117.0           5.128205
    13511                4         104.0           100.0            116.0           5.172414
    13511                5         104.0            99.0            115.0           5.165217
    13511                6         104.0            98.0            114.0           5.157895
    13511               10         104.0    

In [87]:
extreme_rrr = (
    regular_deliveries[
        regular_deliveries["required_run_rate"].notna()
    ]
    .sort_values("required_run_rate", ascending=False)
)

print("Top 15 highest required run rates:")
print(
    extreme_rrr[
        [
            "match_id", "match_format", "season",
            "innings", "cumulative_runs", "chase_target",
            "runs_remaining", "balls_remaining",
            "wickets_remaining", "required_run_rate"
        ]
    ].head(15).to_string(index=False)
)

Top 15 highest required run rates:
 match_id match_format season  innings  cumulative_runs  chase_target  runs_remaining  balls_remaining  wickets_remaining  required_run_rate
    37278         T20I   2024        2              163         298.0           135.0              1.0                  3              810.0
    16663         T20I   2022        2              111         213.0           102.0              1.0                  2              612.0
    16684         T20I   2022        2              119         216.0            97.0              1.0                  1              582.0
    45495         T20I   2025        2               94         189.0            95.0              1.0                  1              570.0
    54213         T20I   2026        2              172         260.0            88.0              1.0                  1              528.0
    37277         T20I   2024        2              135         222.0            87.0              1.0                 

In [88]:
extreme_match_ids = extreme_rrr.head(15)["match_id"].unique()

extreme_match_details = target_matches[
    target_matches["match_id"].isin(extreme_match_ids)
][
    [
        "match_id",
        "match_date",
        "comp_name",
        "home_team",
        "away_team",
        "result",
        "win_team_id"
    ]
]

print(
    extreme_match_details
    .sort_values("match_id")
    .to_string(index=False)
)

 match_id match_date                                   comp_name           home_team           away_team                             result  win_team_id
    16062 2022-07-29       India in West Indies T20I Series 2022         West Indies               India               India Won by 68 Runs         18.0
    16663 2022-09-08                 Mens Twenty20 Asia Cup 2022               India         Afghanistan              India Won by 101 Runs         18.0
    16684 2022-08-13 New Zealand in West Indies T20I Series 2022         West Indies         New Zealand         New Zealand Won by 90 Runs         17.0
    17555 2022-09-23        England in Pakistan T20I Series 2022            Pakistan             England             England Won by 63 Runs         22.0
    17559 2022-10-02        England in Pakistan T20I Series 2022            Pakistan             England             England Won by 67 Runs         22.0
    20184 2023-04-02                                    IPL 2023 Sunrisers Hyderab

In [89]:
rrr_values = regular_deliveries.loc[
    regular_deliveries["required_run_rate"].notna(),
    "required_run_rate"
]

rrr_lower = rrr_values.quantile(0.05)
rrr_upper = rrr_values.quantile(0.95)

print("RRR normalization boundaries:")
print("5th percentile:", rrr_lower)
print("95th percentile:", rrr_upper)

print("\nRRR records:", len(rrr_values))
print("Below lower boundary:", (rrr_values < rrr_lower).sum())
print("Above upper boundary:", (rrr_values > rrr_upper).sum())

RRR normalization boundaries:
5th percentile: 4.199999999999999
95th percentile: 21.214285714285715

RRR records: 91273
Below lower boundary: 4532
Above upper boundary: 4562


In [90]:
# Clip RRR values to the 5th and 95th percentile
regular_deliveries["rrr_clipped"] = (
    regular_deliveries["required_run_rate"]
    .clip(lower=rrr_lower, upper=rrr_upper)
)

# Normalize RRR pressure to a 0–100 scale
regular_deliveries["rrr_pressure_score"] = (
    (regular_deliveries["rrr_clipped"] - rrr_lower)
    / (rrr_upper - rrr_lower)
) * 100

print("RRR pressure score summary:")
print(
    regular_deliveries["rrr_pressure_score"].describe()
)

print("\nMissing RRR pressure scores:",
      regular_deliveries["rrr_pressure_score"].isna().sum())

print("\nSample:")
print(
    regular_deliveries.loc[
        regular_deliveries["innings"] == 2,
        [
            "required_run_rate",
            "rrr_clipped",
            "rrr_pressure_score"
        ]
    ].head(10).to_string(index=False)
)

RRR pressure score summary:
count    91273.000000
mean        36.104805
std         24.203400
min          0.000000
25%         20.478148
50%         32.218915
75%         45.843829
max        100.000000
Name: rrr_pressure_score, dtype: float64

Missing RRR pressure scores: 102715

Sample:
 required_run_rate  rrr_clipped  rrr_pressure_score
          5.243697     5.243697            6.134242
          5.084746     5.084746            5.200017
          5.128205     5.128205            5.455446
          5.172414     5.172414            5.715278
          5.165217     5.165217            5.672982
          5.157895     5.157895            5.629944
          4.991150     4.991150            4.649919
          4.982143     4.982143            4.596977
          5.027027     5.027027            4.860780
          5.018182     5.018182            4.808793


In [91]:
# Wicket pressure: more wickets lost = greater pressure
regular_deliveries["wicket_pressure_score"] = (
    (10 - regular_deliveries["wickets_remaining"]) / 10
) * 100

# Innings-stage pressure: fewer balls remaining = greater pressure
regular_deliveries["time_pressure_score"] = (
    1 - (
        regular_deliveries["balls_remaining"]
        / regular_deliveries["innings_ball_limit"]
    )
) * 100

print("Wicket pressure:")
print(regular_deliveries["wicket_pressure_score"].describe())

print("\nTime pressure:")
print(regular_deliveries["time_pressure_score"].describe())

print("\nMissing wicket pressure:",
      regular_deliveries["wicket_pressure_score"].isna().sum())

print("Missing time pressure:",
      regular_deliveries["time_pressure_score"].isna().sum())

print("\nSample:")
print(
    regular_deliveries[
        [
            "innings", "wickets_remaining",
            "balls_remaining", "wicket_pressure_score",
            "time_pressure_score"
        ]
    ].head(10).to_string(index=False)
)

Wicket pressure:
count    193988.000000
mean         27.321226
std          22.425916
min           0.000000
25%          10.000000
50%          20.000000
75%          40.000000
max         100.000000
Name: wicket_pressure_score, dtype: float64

Time pressure:
count    193988.000000
mean         48.660732
std          28.327738
min           0.000000
25%          24.166667
50%          48.333333
75%          72.500000
max         100.000000
Name: time_pressure_score, dtype: float64

Missing wicket pressure: 0
Missing time pressure: 0

Sample:
 innings  wickets_remaining  balls_remaining  wicket_pressure_score  time_pressure_score
       1                 10            119.0                    0.0             0.833333
       1                 10            118.0                    0.0             1.666667
       1                 10            117.0                    0.0             2.500000
       1                 10            116.0                    0.0             3.333333
      

In [92]:
# Combined pressure score
regular_deliveries["combined_pressure_score"] = np.where(
    regular_deliveries["innings"] == 1,
    (
        regular_deliveries["wicket_pressure_score"]
        + regular_deliveries["time_pressure_score"]
    ) / 2,
    (
        regular_deliveries["rrr_pressure_score"]
        + regular_deliveries["wicket_pressure_score"]
        + regular_deliveries["time_pressure_score"]
    ) / 3
)

# Assign pressure categories
regular_deliveries["pressure_category"] = pd.cut(
    regular_deliveries["combined_pressure_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Low", "Moderate", "High", "Extreme"]
)

print("Combined pressure score:")
print(regular_deliveries["combined_pressure_score"].describe())

print("\nMissing combined scores:",
      regular_deliveries["combined_pressure_score"].isna().sum())

print("\nPressure category distribution:")
print(regular_deliveries["pressure_category"].value_counts().sort_index())

print("\nPressure category percentages:")
print(
    regular_deliveries["pressure_category"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Combined pressure score:
count    193327.000000
mean         37.718894
std          22.531446
min           0.000000
25%          19.583333
50%          34.693912
75%          53.548489
max         100.000000
Name: combined_pressure_score, dtype: float64

Missing combined scores: 661

Pressure category distribution:
pressure_category
Low         66181
Moderate    71471
High        42129
Extreme     13546
Name: count, dtype: int64

Pressure category percentages:
pressure_category
Low         34.23
Moderate    36.97
High        21.79
Extreme      7.01
Name: proportion, dtype: float64


In [93]:
missing_pressure = regular_deliveries[
    regular_deliveries["combined_pressure_score"].isna()
].copy()

print("Missing pressure records by innings:")
print(missing_pressure["innings"].value_counts())

print("\nMissing pressure by reason:")
print(
    missing_pressure.assign(
        reason=np.select(
            [
                missing_pressure["balls_remaining"] == 0,
                missing_pressure["runs_remaining"] <= 0,
                missing_pressure["rrr_pressure_score"].isna()
            ],
            [
                "No balls remaining",
                "Target reached",
                "RRR unavailable"
            ],
            default="Other"
        )
    )["reason"].value_counts()
)

print("\nSample missing records:")
print(
    missing_pressure[
        [
            "match_id", "innings", "cumulative_runs",
            "chase_target", "runs_remaining",
            "balls_remaining", "wickets_remaining",
            "rrr_pressure_score"
        ]
    ].head(15).to_string(index=False)
)

Missing pressure records by innings:
innings
2    661
Name: count, dtype: int64

Missing pressure by reason:
reason
Target reached        394
No balls remaining    267
Name: count, dtype: int64

Sample missing records:
 match_id  innings  cumulative_runs  chase_target  runs_remaining  balls_remaining  wickets_remaining  rrr_pressure_score
    13511        2              104         104.0             0.0             17.0                  9                 NaN
    13512        2              170         172.0             2.0              0.0                  2                 NaN
    13513        2              204         225.0            21.0              0.0                  1                 NaN
    13514        2              159         194.0            35.0              0.0                  5                 NaN
    13522        2              162         158.0            -4.0              7.0                  6                 NaN
    13523        2              178         187.0

In [94]:
# Calculate match state immediately BEFORE each delivery

regular_deliveries["runs_before"] = (
    regular_deliveries["cumulative_runs"]
    - regular_deliveries["total_delivery_runs"]
)

regular_deliveries["legal_balls_before"] = (
    regular_deliveries["cumulative_legal_balls"]
    - regular_deliveries["is_legal_ball"].astype(int)
)

regular_deliveries["wickets_before"] = (
    regular_deliveries["cumulative_wickets"]
    - regular_deliveries["is_match_state_wicket"].astype(int)
)

regular_deliveries["runs_remaining_before"] = (
    regular_deliveries["chase_target"]
    - regular_deliveries["runs_before"]
)

regular_deliveries["balls_remaining_before"] = (
    regular_deliveries["innings_ball_limit"]
    - regular_deliveries["legal_balls_before"]
)

regular_deliveries["wickets_remaining_before"] = (
    10 - regular_deliveries["wickets_before"]
)

print("Pre-delivery state validation:")
print("Negative balls remaining:",
      (regular_deliveries["balls_remaining_before"] < 0).sum())

print("Negative wickets remaining:",
      (regular_deliveries["wickets_remaining_before"] < 0).sum())

print("\nSecond-innings sample:")
print(
    regular_deliveries[
        regular_deliveries["innings"] == 2
    ][
        [
            "cumulative_runs",
            "runs_before",
            "chase_target",
            "runs_remaining_before",
            "balls_remaining_before",
            "wickets_remaining_before"
        ]
    ].head(10).to_string(index=False)
)

Pre-delivery state validation:
Negative balls remaining: 0
Negative wickets remaining: 0

Second-innings sample:
 cumulative_runs  runs_before  chase_target  runs_remaining_before  balls_remaining_before  wickets_remaining_before
               0            0         104.0                  104.0                   120.0                        10
               4            0         104.0                  104.0                   119.0                        10
               4            4         104.0                  100.0                   118.0                        10
               4            4         104.0                  100.0                   117.0                        10
               5            4         104.0                  100.0                   116.0                        10
               6            5         104.0                   99.0                   115.0                        10
              10            6         104.0                   98.0  

In [95]:
# Required run rate immediately before each delivery

regular_deliveries["required_run_rate_before"] = np.where(
    (regular_deliveries["innings"] == 2) &
    (regular_deliveries["balls_remaining_before"] > 0) &
    (regular_deliveries["runs_remaining_before"] > 0),
    (
        regular_deliveries["runs_remaining_before"]
        / regular_deliveries["balls_remaining_before"]
    ) * 6,
    np.nan
)

print("Pre-delivery RRR summary:")
print(
    regular_deliveries["required_run_rate_before"].describe()
)

print("\nMissing RRR:", 
      regular_deliveries["required_run_rate_before"].isna().sum())

print("\nHighest pre-delivery RRR:")
print(
    regular_deliveries.nlargest(
        10, "required_run_rate_before"
    )[
        [
            "match_id",
            "runs_remaining_before",
            "balls_remaining_before",
            "wickets_remaining_before",
            "required_run_rate_before"
        ]
    ].to_string(index=False)
)

Pre-delivery RRR summary:
count    91922.000000
mean        11.405800
std         13.518637
min          0.067416
25%          7.680561
50%          9.662338
75%         12.000000
max        810.000000
Name: required_run_rate_before, dtype: float64

Missing RRR: 102066

Highest pre-delivery RRR:
 match_id  runs_remaining_before  balls_remaining_before  wickets_remaining_before  required_run_rate_before
    37278                  135.0                     1.0                         3                     810.0
    16663                  102.0                     1.0                         2                     612.0
    16684                   97.0                     1.0                         1                     582.0
    45495                   95.0                     1.0                         1                     570.0
    54213                   88.0                     1.0                         1                     528.0
    37277                   87.0                 

In [96]:
# Calculate percentile boundaries using pre-delivery RRR
rrr_values_before = regular_deliveries[
    "required_run_rate_before"
].dropna()

rrr_lower_before = rrr_values_before.quantile(0.05)
rrr_upper_before = rrr_values_before.quantile(0.95)

# Clip extreme values
regular_deliveries["rrr_clipped_before"] = (
    regular_deliveries["required_run_rate_before"]
    .clip(
        lower=rrr_lower_before,
        upper=rrr_upper_before
    )
)

# Normalize to 0–100
regular_deliveries["rrr_pressure_before"] = (
    (
        regular_deliveries["rrr_clipped_before"]
        - rrr_lower_before
    )
    / (rrr_upper_before - rrr_lower_before)
) * 100

print("5th percentile:", rrr_lower_before)
print("95th percentile:", rrr_upper_before)

print("\nPre-delivery RRR pressure:")
print(regular_deliveries["rrr_pressure_before"].describe())

print("\nMissing scores:",
      regular_deliveries["rrr_pressure_before"].isna().sum())

5th percentile: 4.222222222222222
95th percentile: 21.0

Pre-delivery RRR pressure:
count    91922.000000
mean        36.260590
std         24.193989
min          0.000000
25%         20.612614
50%         32.424529
75%         46.357616
max        100.000000
Name: rrr_pressure_before, dtype: float64

Missing scores: 102066


In [97]:
# Pre-delivery wicket pressure
regular_deliveries["wicket_pressure_before"] = (
    (10 - regular_deliveries["wickets_remaining_before"]) / 10
) * 100

# Pre-delivery time pressure
regular_deliveries["time_pressure_before"] = (
    1 - (
        regular_deliveries["balls_remaining_before"]
        / regular_deliveries["innings_ball_limit"]
    )
) * 100

print("Wicket pressure before delivery:")
print(regular_deliveries["wicket_pressure_before"].describe())

print("\nTime pressure before delivery:")
print(regular_deliveries["time_pressure_before"].describe())

print("\nMissing wicket pressure:",
      regular_deliveries["wicket_pressure_before"].isna().sum())

print("Missing time pressure:",
      regular_deliveries["time_pressure_before"].isna().sum())

Wicket pressure before delivery:
count    193988.000000
mean         26.781090
std          22.188568
min           0.000000
25%          10.000000
50%          20.000000
75%          40.000000
max          90.000000
Name: wicket_pressure_before, dtype: float64

Time pressure before delivery:
count    193988.000000
mean         47.854778
std          28.333604
min           0.000000
25%          23.333333
50%          47.500000
75%          71.666667
max          99.166667
Name: time_pressure_before, dtype: float64

Missing wicket pressure: 0
Missing time pressure: 0


In [98]:
# Pre-delivery combined pressure score
regular_deliveries["pressure_score"] = np.where(
    regular_deliveries["innings"] == 1,
    (
        regular_deliveries["wicket_pressure_before"]
        + regular_deliveries["time_pressure_before"]
    ) / 2,
    (
        regular_deliveries["rrr_pressure_before"]
        + regular_deliveries["wicket_pressure_before"]
        + regular_deliveries["time_pressure_before"]
    ) / 3
)

# Assign pressure categories
regular_deliveries["pressure_category"] = pd.cut(
    regular_deliveries["pressure_score"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Low", "Moderate", "High", "Extreme"]
)

print("Pressure score summary:")
print(regular_deliveries["pressure_score"].describe())

print("\nMissing pressure scores:",
      regular_deliveries["pressure_score"].isna().sum())

print("\nPressure category distribution:")
print(
    regular_deliveries["pressure_category"]
    .value_counts()
    .sort_index()
)

print("\nPressure percentages:")
print(
    regular_deliveries["pressure_category"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Pressure score summary:
count    193976.000000
mean         37.253821
std          22.458830
min           0.000000
25%          19.166667
50%          34.323105
75%          52.942644
max          96.388889
Name: pressure_score, dtype: float64

Missing pressure scores: 12

Pressure category distribution:
pressure_category
Low         67678
Moderate    71448
High        41892
Extreme     12958
Name: count, dtype: int64

Pressure percentages:
pressure_category
Low         34.89
Moderate    36.83
High        21.60
Extreme      6.68
Name: proportion, dtype: float64


In [99]:
missing_pressure_before = regular_deliveries[
    regular_deliveries["pressure_score"].isna()
].copy()

print("Missing records by innings:")
print(missing_pressure_before["innings"].value_counts())

print("\nMissing record details:")
print(
    missing_pressure_before[
        [
            "match_id",
            "match_format",
            "innings",
            "runs_before",
            "runs_remaining_before",
            "balls_remaining_before",
            "wickets_remaining_before",
            "rrr_pressure_before"
        ]
    ].to_string(index=False)
)

Missing records by innings:
innings
2    12
Name: count, dtype: int64

Missing record details:
 match_id match_format  innings  runs_before  runs_remaining_before  balls_remaining_before  wickets_remaining_before  rrr_pressure_before
    48610         T20I        2          115                    0.0                    12.0                         7                  NaN
    48610         T20I        2          115                    0.0                    11.0                         7                  NaN
    48610         T20I        2          116                   -1.0                    10.0                         7                  NaN
    48610         T20I        2          120                   -5.0                     9.0                         7                  NaN
    48610         T20I        2          120                   -5.0                     8.0                         7                  NaN
    48610         T20I        2          120                   -5.0    

In [100]:
print(
    target_matches[
        target_matches["match_id"] == 48610
    ][
        [
            "match_id",
            "match_date",
            "comp_name",
            "home_team",
            "away_team",
            "result",
            "win_team_id"
        ]
    ].to_string(index=False)
)

 match_id match_date                             comp_name home_team away_team                              result  win_team_id
    48610 2026-01-30 England in Sri Lanka T20I Series 2026 Sri Lanka   England England Won by 11 runs (DLS method)         22.0


In [101]:
# Keep deliveries with valid pre-delivery pressure scores
active_pressure_deliveries = regular_deliveries[
    regular_deliveries["pressure_score"].notna()
].copy()

print("Original deliveries:", len(regular_deliveries))
print("Active pressure deliveries:", len(active_pressure_deliveries))
print("Excluded deliveries:", len(regular_deliveries) - len(active_pressure_deliveries))

print("\nPressure categories:")
print(
    active_pressure_deliveries["pressure_category"]
    .value_counts()
    .sort_index()
)

print("\nMissing pressure scores:",
      active_pressure_deliveries["pressure_score"].isna().sum())

Original deliveries: 193988
Active pressure deliveries: 193976
Excluded deliveries: 12

Pressure categories:
pressure_category
Low         67678
Moderate    71448
High        41892
Extreme     12958
Name: count, dtype: int64

Missing pressure scores: 0


In [102]:
print("Pressure distribution by format:")

print(
    pd.crosstab(
        active_pressure_deliveries["match_format"],
        active_pressure_deliveries["pressure_category"],
        margins=True
    )
)

print("\nPressure percentages by format:")

print(
    pd.crosstab(
        active_pressure_deliveries["match_format"],
        active_pressure_deliveries["pressure_category"],
        normalize="index"
    ).mul(100).round(2)
)

print("\nPressure distribution by innings:")

print(
    pd.crosstab(
        active_pressure_deliveries["innings"],
        active_pressure_deliveries["pressure_category"],
        margins=True
    )
)

Pressure distribution by format:
pressure_category    Low  Moderate   High  Extreme     All
match_format                                              
T20                30104     33327  18753     5489   87673
T20I               37574     38121  23139     7469  106303
All                67678     71448  41892    12958  193976

Pressure percentages by format:
pressure_category    Low  Moderate   High  Extreme
match_format                                      
T20                34.34     38.01  21.39     6.26
T20I               35.35     35.86  21.77     7.03

Pressure distribution by innings:
pressure_category    Low  Moderate   High  Extreme     All
innings                                                   
1                  35595     32704  27350     6405  102054
2                  32083     38744  14542     6553   91922
All                67678     71448  41892    12958  193976


In [103]:
# Identify dismissals credited to the striker
active_pressure_deliveries["is_striker_dismissed"] = (
    active_pressure_deliveries["is_match_state_wicket"].eq(True)
    & active_pressure_deliveries["dismissed_player_id"].eq(
        active_pressure_deliveries["striker_id"]
    )
)

# Group batter performance by pressure category
batter_pressure_stats = (
    active_pressure_deliveries
    .groupby(
        ["striker_id", "striker_name", "pressure_category"],
        observed=True
    )
    .agg(
        delivery_records=("match_id", "size"),
        balls_faced=("is_ball_faced", "sum"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_striker_dismissed", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Calculate performance metrics
batter_pressure_stats["strike_rate"] = (
    batter_pressure_stats["batter_runs"]
    / batter_pressure_stats["balls_faced"].replace(0, np.nan)
) * 100

batter_pressure_stats["dismissal_rate"] = (
    batter_pressure_stats["dismissals"]
    / batter_pressure_stats["balls_faced"].replace(0, np.nan)
) * 100

print("Batter pressure statistics:", batter_pressure_stats.shape)

print("\nPressure categories:")
print(batter_pressure_stats["pressure_category"].value_counts())

print("\nSample:")
print(batter_pressure_stats.head(15).to_string(index=False))

Batter pressure statistics: (2056, 12)

Pressure categories:
pressure_category
High        617
Extreme     546
Moderate    511
Low         382
Name: count, dtype: int64

Sample:
 striker_id         striker_name pressure_category  delivery_records  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate  dismissal_rate
          8 Mahendra Singh Dhoni          Moderate                37           35           20      1      0           0               9    57.142857        0.000000
          8 Mahendra Singh Dhoni              High               287          273          338     23     17          10              39   123.809524        3.663004
          8 Mahendra Singh Dhoni           Extreme               173          155          335     26     28          12              25   216.129032        7.741935
         23        Piyush Chawla          Moderate                 3            3            2      0      0           0               1    66.666667        0

In [104]:
# Combine pressure categories into two groups

active_pressure_deliveries["pressure_group"] = (
    active_pressure_deliveries["pressure_category"]
    .map({
        "Low": "Normal",
        "Moderate": "Normal",
        "High": "High",
        "Extreme": "High"
    })
)

print("Pressure group distribution:")
print(
    active_pressure_deliveries["pressure_group"]
    .value_counts()
)

print("\nPressure percentages:")
print(
    active_pressure_deliveries["pressure_group"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Pressure group distribution:
pressure_group
Normal    139126
High       54850
Name: count, dtype: int64

Pressure percentages:
pressure_group
Normal    71.72
High      28.28
Name: proportion, dtype: float64


In [105]:
batter_group_stats = (
    active_pressure_deliveries
    .groupby(
        ["striker_id", "striker_name", "pressure_group"],
        observed=True
    )
    .agg(
        balls_faced=("is_ball_faced", "sum"),
        batter_runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_striker_dismissed", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Calculate strike rate
batter_group_stats["strike_rate"] = (
    batter_group_stats["batter_runs"]
    / batter_group_stats["balls_faced"].replace(0, np.nan)
) * 100

# Calculate dismissal rate
batter_group_stats["dismissal_rate"] = (
    batter_group_stats["dismissals"]
    / batter_group_stats["balls_faced"].replace(0, np.nan)
) * 100

print("Batter group statistics:", batter_group_stats.shape)

print("\nSample:")
print(batter_group_stats.head(15).to_string(index=False))

print("\nRecords by pressure group:")
print(batter_group_stats["pressure_group"].value_counts())

Batter group statistics: (1255, 11)

Sample:
 striker_id         striker_name pressure_group  balls_faced  batter_runs  fours  sixes  dismissals  matches_played  strike_rate  dismissal_rate
          8 Mahendra Singh Dhoni           High          428          673     49     45          22              46   157.242991        5.140187
          8 Mahendra Singh Dhoni         Normal           35           20      1      0           0               9    57.142857        0.000000
         23        Piyush Chawla           High           34           38      2      2           6              10   111.764706       17.647059
         23        Piyush Chawla         Normal            3            2      0      0           0               1    66.666667        0.000000
         28        Ishant Sharma           High            8            2      0      0           0               3    25.000000        0.000000
         47          Virat Kohli           High          146          282     24     

In [106]:
# Pivot pressure-group statistics
batter_comparison = batter_group_stats.pivot(
    index=["striker_id", "striker_name"],
    columns="pressure_group",
    values=[
        "balls_faced",
        "batter_runs",
        "dismissals",
        "matches_played",
        "strike_rate",
        "dismissal_rate"
    ]
)

batter_comparison.columns = [
    f"{metric}_{group}"
    for metric, group in batter_comparison.columns
]

batter_comparison = batter_comparison.reset_index()

# Apply minimum sample-size requirements
eligible_batters = batter_comparison[
    (batter_comparison["balls_faced_Normal"] >= 30) &
    (batter_comparison["balls_faced_High"] >= 30) &
    (batter_comparison["matches_played_High"] >= 5)
].copy()

print("Total unique batters:", batter_comparison.shape[0])
print("Eligible batters:", eligible_batters.shape[0])

print("\nSample eligible batters:")
print(
    eligible_batters[
        [
            "striker_name",
            "balls_faced_Normal",
            "balls_faced_High",
            "matches_played_High",
            "strike_rate_Normal",
            "strike_rate_High"
        ]
    ].head(15).to_string(index=False)
)

Total unique batters: 824
Eligible batters: 175

Sample eligible batters:
        striker_name  balls_faced_Normal  balls_faced_High  matches_played_High  strike_rate_Normal  strike_rate_High
Mahendra Singh Dhoni                35.0             428.0                 46.0           57.142857        157.242991
         Virat Kohli              2310.0             146.0                 14.0          142.207792        193.150685
      Faf du Plessis              1204.0              72.0                  8.0          143.189369        158.333333
        Rohit Sharma              1560.0              59.0                  5.0          141.602564        203.389831
      Shikhar Dhawan               678.0              79.0                  7.0          125.958702        165.822785
      Dinesh Karthik                57.0             582.0                 53.0          163.157895        167.697595
     Ravindra Jadeja               341.0             617.0                 53.0          119.061584 

In [107]:
# Strike-rate improvement under pressure
eligible_batters["strike_rate_change"] = (
    eligible_batters["strike_rate_High"]
    - eligible_batters["strike_rate_Normal"]
)

eligible_batters["strike_rate_change_pct"] = np.where(
    eligible_batters["strike_rate_Normal"] > 0,
    (
        eligible_batters["strike_rate_change"]
        / eligible_batters["strike_rate_Normal"]
    ) * 100,
    np.nan
)

# Change in dismissal rate
eligible_batters["dismissal_rate_change"] = (
    eligible_batters["dismissal_rate_High"]
    - eligible_batters["dismissal_rate_Normal"]
)

# High-pressure scoring contribution
eligible_batters["high_pressure_runs_per_match"] = (
    eligible_batters["batter_runs_High"]
    / eligible_batters["matches_played_High"]
)

print("Performance comparison:")
print(
    eligible_batters[
        [
            "striker_name",
            "strike_rate_Normal",
            "strike_rate_High",
            "strike_rate_change_pct",
            "dismissal_rate_change",
            "high_pressure_runs_per_match"
        ]
    ].head(20).to_string(index=False)
)

Performance comparison:
        striker_name  strike_rate_Normal  strike_rate_High  strike_rate_change_pct  dismissal_rate_change  high_pressure_runs_per_match
Mahendra Singh Dhoni           57.142857        157.242991              175.175234               5.140187                     14.630435
         Virat Kohli          142.207792        193.150685               35.822856               1.122576                     20.142857
      Faf du Plessis          143.189369        158.333333               10.576179               4.595792                     14.250000
        Rohit Sharma          141.602564        203.389831               43.634285              -1.930682                     24.000000
      Shikhar Dhawan          125.958702        165.822785               31.648534               1.818453                     18.714286
      Dinesh Karthik          163.157895        167.697595                2.782397               0.063303                     18.415094
     Ravindra Jadeja    

In [108]:
print("High-pressure balls distribution:")
print(
    eligible_batters["balls_faced_High"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nEligible batters at different sample thresholds:")

for threshold in [30, 50, 75, 100, 150]:
    count = (
        eligible_batters["balls_faced_High"] >= threshold
    ).sum()

    print(f"At least {threshold} high-pressure balls: {count}")

High-pressure balls distribution:
count    175.000000
mean     203.788571
std      177.727565
min       35.000000
10%       54.400000
25%       74.000000
50%      146.000000
75%      279.000000
90%      451.200000
max      857.000000
Name: balls_faced_High, dtype: float64

Eligible batters at different sample thresholds:
At least 30 high-pressure balls: 175
At least 50 high-pressure balls: 159
At least 75 high-pressure balls: 130
At least 100 high-pressure balls: 109
At least 150 high-pressure balls: 81


In [109]:
# Apply stricter sample-size requirements
final_batter_pool = eligible_batters[
    (eligible_batters["balls_faced_Normal"] >= 50) &
    (eligible_batters["balls_faced_High"] >= 50) &
    (eligible_batters["matches_played_High"] >= 5)
].copy()

print("Total eligible batters:", len(final_batter_pool))

print("\nSample-size validation:")
print("Minimum normal balls:",
      final_batter_pool["balls_faced_Normal"].min())

print("Minimum high-pressure balls:",
      final_batter_pool["balls_faced_High"].min())

print("Minimum high-pressure matches:",
      final_batter_pool["matches_played_High"].min())

print("\nTop 15 by high-pressure runs:")
print(
    final_batter_pool[
        [
            "striker_name",
            "batter_runs_High",
            "balls_faced_High",
            "strike_rate_High",
            "dismissal_rate_High",
            "matches_played_High"
        ]
    ]
    .sort_values("batter_runs_High", ascending=False)
    .head(15)
    .to_string(index=False)
)

Total eligible batters: 151

Sample-size validation:
Minimum normal balls: 50.0
Minimum high-pressure balls: 53.0
Minimum high-pressure matches: 5.0

Top 15 by high-pressure runs:
        striker_name  batter_runs_High  balls_faced_High  strike_rate_High  dismissal_rate_High  matches_played_High
           Tim David            1377.0             749.0        183.845127             5.473965                 69.0
       Hardik Pandya            1375.0             857.0        160.443407             5.834306                 71.0
         Rinku Singh            1320.0             757.0        174.372523             4.359313                 60.0
       Rovman Powell            1249.0             729.0        171.330590             6.172840                 59.0
Namboori Tilak Varma            1241.0             701.0        177.032810             5.135521                 61.0
         Shivam Dube            1080.0             638.0        169.278997             5.956113                 60.0
 

In [110]:
batter_metrics = final_batter_pool[
    [
        "strike_rate_High",
        "strike_rate_change_pct",
        "dismissal_rate_High",
        "matches_played_High",
        "batter_runs_High"
    ]
]

print("Correlation between batter performance metrics:")
print(batter_metrics.corr().round(3))

Correlation between batter performance metrics:
                        strike_rate_High  strike_rate_change_pct  \
strike_rate_High                   1.000                   0.259   
strike_rate_change_pct             0.259                   1.000   
dismissal_rate_High               -0.073                  -0.203   
matches_played_High                0.073                  -0.055   
batter_runs_High                   0.239                  -0.001   

                        dismissal_rate_High  matches_played_High  \
strike_rate_High                     -0.073                0.073   
strike_rate_change_pct               -0.203               -0.055   
dismissal_rate_High                   1.000               -0.045   
matches_played_High                  -0.045                1.000   
batter_runs_High                     -0.204                0.939   

                        batter_runs_High  
strike_rate_High                   0.239  
strike_rate_change_pct            -0.001  
dismi

In [111]:
print("Eligible batter pool:", len(final_batter_pool))

print("\nMetric ranges:")
print(
    final_batter_pool[
        [
            "strike_rate_High",
            "strike_rate_change_pct",
            "dismissal_rate_High",
            "matches_played_High"
        ]
    ].describe().round(2)
)

Eligible batter pool: 151

Metric ranges:
       strike_rate_High  strike_rate_change_pct  dismissal_rate_High  \
count            151.00                  151.00               151.00   
mean             161.55                   29.13                 6.94   
std               21.62                   22.32                 2.24   
min              110.00                  -16.81                 1.25   
25%              146.79                   14.94                 5.50   
50%              161.73                   28.40                 6.80   
75%              176.53                   40.21                 8.00   
max              215.33                  139.73                14.86   

       matches_played_High  
count               151.00  
mean                 22.28  
std                  16.60  
min                   5.00  
25%                   9.00  
50%                  17.00  
75%                  30.00  
max                  79.00  


In [112]:
# Convert each metric into a percentile score from 0 to 100.

final_batter_pool["sr_score"] = (
    final_batter_pool["strike_rate_High"].rank(pct=True) * 100
)

final_batter_pool["improvement_score"] = (
    final_batter_pool["strike_rate_change_pct"].rank(pct=True) * 100
)

# Lower dismissal rate is better, so reverse the ranking.
final_batter_pool["survival_score"] = (
    final_batter_pool["dismissal_rate_High"].rank(ascending=False, pct=True) * 100
)

final_batter_pool["experience_score"] = (
    final_batter_pool["matches_played_High"].rank(pct=True) * 100
)

print(
    final_batter_pool[
        [
            "striker_name",
            "sr_score",
            "improvement_score",
            "survival_score",
            "experience_score"
        ]
    ].head(10).round(2).to_string(index=False)
)

       striker_name  sr_score  improvement_score  survival_score  experience_score
        Virat Kohli     93.38              66.23           91.72             41.06
     Faf du Plessis     42.38              18.54           21.85             19.21
       Rohit Sharma     98.68              79.47           96.03              3.31
     Shikhar Dhawan     56.95              58.94           82.78             12.91
     Dinesh Karthik     60.93              10.60           80.79             93.05
    Ravindra Jadeja     23.18              35.76           86.09             93.05
      Manish Pandey     26.49              57.62           79.14              8.28
       David Warner      7.95               1.99           58.94             12.91
Ravichandran Ashwin     38.08              68.21            7.95             68.87
     Kieron Pollard      9.93              98.01           80.13             41.06


In [113]:
final_batter_pool["batter_reliability_score"] = (
    0.35 * final_batter_pool["sr_score"]
    + 0.25 * final_batter_pool["improvement_score"]
    + 0.25 * final_batter_pool["survival_score"]
    + 0.15 * final_batter_pool["experience_score"]
)

batter_rankings = (
    final_batter_pool
    .sort_values("batter_reliability_score", ascending=False)
    .reset_index(drop=True)
)

batter_rankings["rank"] = batter_rankings.index + 1

print("TOP 15 BATTERS UNDER PRESSURE")
print(
    batter_rankings[
        [
            "rank",
            "striker_name",
            "batter_reliability_score",
            "strike_rate_High",
            "strike_rate_change_pct",
            "dismissal_rate_High",
            "matches_played_High",
            "balls_faced_High"
        ]
    ].head(15).round(2).to_string(index=False)
)

TOP 15 BATTERS UNDER PRESSURE
 rank          striker_name  batter_reliability_score  strike_rate_High  strike_rate_change_pct  dismissal_rate_High  matches_played_High  balls_faced_High
    1        Tristan Stubbs                     85.03            180.00                   53.45                 4.86                 45.0             555.0
    2           Rinku Singh                     82.33            174.37                   44.03                 4.36                 60.0             757.0
    3            Babar Azam                     81.11            210.75                   66.52                 5.38                  8.0              93.0
    4       Ruturaj Gaikwad                     80.99            215.33                   62.12                 5.84                 12.0             137.0
    5          David Miller                     79.54            176.13                   38.17                 5.05                 55.0             574.0
    6          Rohit Sharma       

In [114]:
batter_rankings["robust_sample"] = (
    (batter_rankings["balls_faced_High"] >= 100) &
    (batter_rankings["matches_played_High"] >= 10)
)

print("TOP 15 — SAMPLE-SIZE ROBUSTNESS CHECK")

print(
    batter_rankings[
        [
            "rank",
            "striker_name",
            "batter_reliability_score",
            "balls_faced_High",
            "matches_played_High",
            "robust_sample"
        ]
    ].head(15).round(2).to_string(index=False)
)

print(
    "\nPlayers meeting stricter sample requirements:",
    batter_rankings["robust_sample"].sum()
)

TOP 15 — SAMPLE-SIZE ROBUSTNESS CHECK
 rank          striker_name  batter_reliability_score  balls_faced_High  matches_played_High  robust_sample
    1        Tristan Stubbs                     85.03             555.0                 45.0           True
    2           Rinku Singh                     82.33             757.0                 60.0           True
    3            Babar Azam                     81.11              93.0                  8.0          False
    4       Ruturaj Gaikwad                     80.99             137.0                 12.0           True
    5          David Miller                     79.54             574.0                 55.0           True
    6          Rohit Sharma                     78.91              59.0                  5.0          False
    7           Virat Kohli                     78.33             146.0                 14.0           True
    8  Namboori Tilak Varma                     77.91             701.0                 61.0      

In [115]:
robust_batter_pool = batter_rankings[
    batter_rankings["robust_sample"]
].copy()

robust_batter_rankings = (
    robust_batter_pool
    .sort_values("batter_reliability_score", ascending=False)
    .reset_index(drop=True)
)

robust_batter_rankings["final_rank"] = (
    robust_batter_rankings.index + 1
)

print("ROBUST BATTER RANKING — TOP 15")

print(
    robust_batter_rankings[
        [
            "final_rank",
            "striker_name",
            "batter_reliability_score",
            "strike_rate_High",
            "strike_rate_change_pct",
            "dismissal_rate_High",
            "balls_faced_High",
            "matches_played_High"
        ]
    ].head(15).round(2).to_string(index=False)
)

print("\nRobust eligible batters:", len(robust_batter_rankings))

ROBUST BATTER RANKING — TOP 15
 final_rank         striker_name  batter_reliability_score  strike_rate_High  strike_rate_change_pct  dismissal_rate_High  balls_faced_High  matches_played_High
          1       Tristan Stubbs                     85.03            180.00                   53.45                 4.86             555.0                 45.0
          2          Rinku Singh                     82.33            174.37                   44.03                 4.36             757.0                 60.0
          3      Ruturaj Gaikwad                     80.99            215.33                   62.12                 5.84             137.0                 12.0
          4         David Miller                     79.54            176.13                   38.17                 5.05             574.0                 55.0
          5          Virat Kohli                     78.33            193.15                   35.82                 4.11             146.0                 14.0
   

In [116]:
# Compare high-pressure batting exposure across IPL and T20I.

format_batter_stats = (
    active_pressure_deliveries[
        active_pressure_deliveries["pressure_group"] == "High"
    ]
    .groupby(
        ["striker_id", "striker_name", "match_format"],
        observed=True
    )
    .agg(
        balls_faced=("is_ball_faced", "sum"),
        runs=("runs", "sum"),
        matches=("match_id", "nunique"),
        dismissals=("is_striker_dismissed", "sum")
    )
    .reset_index()
)

top_15_ids = robust_batter_rankings.head(15)["striker_id"]

print("HIGH-PRESSURE PERFORMANCE BY FORMAT — TOP 15")

print(
    format_batter_stats[
        format_batter_stats["striker_id"].isin(top_15_ids)
    ]
    .sort_values(["striker_name", "match_format"])
    .to_string(index=False)
)

HIGH-PRESSURE PERFORMANCE BY FORMAT — TOP 15
 striker_id         striker_name match_format  balls_faced  runs  matches  dismissals
       1077         David Miller          T20          396   666       40          20
       1077         David Miller         T20I          178   345       15           9
       4260     Heinrich Klaasen          T20          443   832       32          25
       4260     Heinrich Klaasen         T20I           59   113        7           4
       4773       Jaker Ali Anik         T20I          272   439       20          13
       2639       Marcus Stoinis          T20          322   625       33          22
       2639       Marcus Stoinis         T20I          132   203       10           7
        796         Matthew Wade         T20I          229   371       19           7
       3024      Mohammad Rizwan         T20I          106   193       10           6
      37052           Naman Dhir          T20          182   333       15          10
      370

In [117]:
format_batter_stats["strike_rate"] = (
    format_batter_stats["runs"]
    / format_batter_stats["balls_faced"]
    * 100
)

print("FORMAT-WISE HIGH-PRESSURE STRIKE RATES — TOP 15")

print(
    format_batter_stats[
        format_batter_stats["striker_id"].isin(top_15_ids)
    ][
        [
            "striker_name",
            "match_format",
            "balls_faced",
            "runs",
            "strike_rate",
            "matches",
            "dismissals"
        ]
    ]
    .sort_values(["striker_name", "match_format"])
    .round(2)
    .to_string(index=False)
)

FORMAT-WISE HIGH-PRESSURE STRIKE RATES — TOP 15
        striker_name match_format  balls_faced  runs  strike_rate  matches  dismissals
        David Miller          T20          396   666       168.18       40          20
        David Miller         T20I          178   345       193.82       15           9
    Heinrich Klaasen          T20          443   832       187.81       32          25
    Heinrich Klaasen         T20I           59   113       191.53        7           4
      Jaker Ali Anik         T20I          272   439       161.40       20          13
      Marcus Stoinis          T20          322   625       194.10       33          22
      Marcus Stoinis         T20I          132   203       153.79       10           7
        Matthew Wade         T20I          229   371       162.01       19           7
     Mohammad Rizwan         T20I          106   193       182.08       10           6
          Naman Dhir          T20          182   333       182.97       15        

In [118]:
format_pivot = format_batter_stats.pivot(
    index=["striker_id", "striker_name"],
    columns="match_format",
    values=["balls_faced", "strike_rate", "matches"]
)

format_pivot.columns = [
    f"{metric}_{fmt}" for metric, fmt in format_pivot.columns
]

format_pivot = format_pivot.reset_index()

format_consistency = format_pivot[
    (format_pivot["balls_faced_T20"] >= 50) &
    (format_pivot["balls_faced_T20I"] >= 50)
].copy()

format_consistency["sr_difference"] = (
    format_consistency["strike_rate_T20"]
    - format_consistency["strike_rate_T20I"]
).abs()

print("PLAYERS WITH MEANINGFUL HIGH-PRESSURE EXPOSURE IN BOTH FORMATS")

print(
    format_consistency[
        [
            "striker_name",
            "balls_faced_T20",
            "balls_faced_T20I",
            "strike_rate_T20",
            "strike_rate_T20I",
            "sr_difference"
        ]
    ]
    .sort_values("sr_difference")
    .round(2)
    .to_string(index=False)
)

print("\nPlayers meeting the 50-ball threshold in both formats:",
      len(format_consistency))

PLAYERS WITH MEANINGFUL HIGH-PRESSURE EXPOSURE IN BOTH FORMATS
        striker_name  balls_faced_T20  balls_faced_T20I  strike_rate_T20  strike_rate_T20I  sr_difference
           Tim David            504.0             245.0           184.33            182.86           1.47
  Azmatullah Omarzai             63.0             303.0           165.08            162.71           2.37
    Heinrich Klaasen            443.0              59.0           187.81            191.53           3.72
        Harshit Rana             50.0              75.0           112.00            116.00           4.00
       Glenn Maxwell             57.0             103.0           177.19            181.55           4.36
 Sherfane Rutherford            131.0             218.0           170.99            164.22           6.77
        Rishabh Pant            213.0              70.0           191.55            184.29           7.26
        Deepak Hooda            110.0              56.0           133.64            141.0

In [119]:
# Save batter analysis outputs

final_batter_pool.to_csv(
    OUTPUT_PATH + "batter_pool_with_scores.csv",
    index=False
)

robust_batter_rankings.to_csv(
    OUTPUT_PATH + "robust_batter_rankings.csv",
    index=False
)

format_batter_stats.to_csv(
    OUTPUT_PATH + "batter_format_performance.csv",
    index=False
)

format_consistency.to_csv(
    OUTPUT_PATH + "batter_format_consistency.csv",
    index=False
)

print("Batter analysis files saved successfully:")
print(os.listdir(OUTPUT_PATH))

Batter analysis files saved successfully:
['batter_basic_stats.csv', 'batter_format_consistency.csv', 'batter_format_performance.csv', 'batter_pool_with_scores.csv', 'batter_pressure_category_stats.csv', 'eligible_batter_reliability_stats.csv', 'robust_batter_rankings.csv']
